# Liquid JEPA: Liquid Networks vs Diffusion on Push-T Trajectory Generation

**Shared Backbone Strategy for Fair Comparison**

This notebook implements a controlled comparison between **liquid neural networks** and **diffusion models** for trajectory generation on the **Push-T dataset**.

## Key Design Principles

1. **Frozen Encoders**: Both models use the same frozen CLIP image encoder
2. **Shared Backbone**: Identical Transformer backbone for conditioning and feature fusion
3. **Identical Data & Splits**: Same dataset, preprocessing, train/val/test splits
4. **Isolated Comparison**: Only the trajectory generator differs (liquid vs diffusion)

This ensures that performance differences reflect core modeling differences, not encoder capacity or data handling.

## Quick Start

- **Current Mode**: 3 epochs per model on real Push-T data (deployment sanity run)
- **Production Mode**: Change `NUM_EPOCHS=3` to `NUM_EPOCHS=50` in training setup

## Roadmap

| Section | Goal |
|---------|------|
| 1 | Load frozen CLIP encoder |
| 2 | Load & preprocess Push-T zarr dataset |
| 3 | Define shared Transformer backbone |
| 4 | Implement diffusion trajectory generator |
| 5 | Implement liquid net trajectory generator |
| 6 | Unified training pipeline |
| 7 | Evaluation on test split |
| 8 | Visualization & comparison |

## 1. Import Libraries and Load Frozen CLIP Encoders

In [ ]:
import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

# Install required packages (including zarr for Push-T dataset)
pip_install(
    "torch", "torchvision", "transformers", "clip",
    "numpy", "matplotlib", "seaborn", "pandas", "tqdm",
    "diffusers", "einops", "zarr"
)
print("Dependencies installed.")

import os
import math
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tqdm.auto import tqdm
from typing import Dict, Tuple, Optional, List
import zarr

print(f"PyTorch {torch.__version__} | Device: {torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')}")

In [ ]:
# Load pretrained CLIP encoders (frozen for both models)
try:
    import clip
except ImportError:
    # If the clip package installed is not OpenAI's, install it
    pip_install("git+https://github.com/openai/CLIP.git")
    import clip

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Load CLIP ViT-B/32 as the shared vision and language encoder
try:
    clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
except (RuntimeError, AttributeError) as e:
    print(f"CLIP loading warning (expected on first run): {e}")
    print("Using deterministic mock CLIP encoders for demonstration...")

    class MockCLIPModel(nn.Module):
        """Deterministic mock CLIP used only when the real model is unavailable."""
        def __init__(self, embed_dim: int = 512):
            super().__init__()
            self.embed_dim = embed_dim
            self.image_pool = nn.AdaptiveAvgPool2d((8, 8))
            self.image_proj = nn.Linear(8 * 8 * 3, embed_dim, bias=False)
            self.text_proj = nn.Linear(77, embed_dim, bias=False)

            with torch.no_grad():
                torch.manual_seed(7)
                nn.init.normal_(self.image_proj.weight, std=0.02)
                nn.init.normal_(self.text_proj.weight, std=0.02)

            for param in self.parameters():
                param.requires_grad = False

        def encode_image(self, image):
            image = image.float()
            pooled = self.image_pool(image).flatten(1)
            return F.normalize(self.image_proj(pooled), dim=-1)

        def encode_text(self, text):
            text = text.float()
            if text.ndim == 1:
                text = text.unsqueeze(0)
            if text.shape[-1] != 77:
                text = F.pad(text, (0, max(0, 77 - text.shape[-1])))[:, :77]
            return F.normalize(self.text_proj(text), dim=-1)

    clip_model = MockCLIPModel().to(device)
    clip_preprocess = None

clip_model.eval()

# Freeze CLIP weights — both diffusion and liquid will use this
for param in clip_model.parameters():
    param.requires_grad = False

# Extract component dimensions
CLIP_FEAT_DIM = 512  # ViT-B/32 has 512-dim embeddings
print(f"CLIP feature dimension: {CLIP_FEAT_DIM}")
print("✓ Frozen CLIP encoders loaded and frozen")

## 2. Load and Preprocess Push-T Dataset

This section loads real Push-T trajectories from zarr, normalizes action/state, and returns raw image sequences for frozen CLIP encoding in the shared backbone.

In [ ]:
# ---- Push-T Dataset Utilities ----
def create_sample_indices(episode_ends: np.ndarray, sequence_length: int,
                         pad_before: int = 0, pad_after: int = 0):
    """Create valid sample indices from episode boundaries."""
    indices = []
    for i in range(len(episode_ends)):
        start_idx = 0
        if i > 0:
            start_idx = episode_ends[i - 1]
        end_idx = episode_ends[i]
        episode_length = end_idx - start_idx

        min_start = -pad_before
        max_start = episode_length - sequence_length + pad_after

        for idx in range(min_start, max_start + 1):
            buffer_start_idx = max(idx, 0) + start_idx
            buffer_end_idx = min(idx + sequence_length, episode_length) + start_idx
            start_offset = buffer_start_idx - (idx + start_idx)
            end_offset = (idx + sequence_length + start_idx) - buffer_end_idx
            sample_start_idx = 0 + start_offset
            sample_end_idx = sequence_length - end_offset
            indices.append([buffer_start_idx, buffer_end_idx,
                          sample_start_idx, sample_end_idx])
    return np.array(indices)

def sample_sequence(train_data, sequence_length,
                   buffer_start_idx, buffer_end_idx,
                   sample_start_idx, sample_end_idx):
    """Extract sequence with padding."""
    result = {}
    for key, input_arr in train_data.items():
        sample = input_arr[buffer_start_idx:buffer_end_idx]
        data = sample
        if (sample_start_idx > 0) or (sample_end_idx < sequence_length):
            data = np.zeros(
                shape=(sequence_length,) + input_arr.shape[1:],
                dtype=input_arr.dtype)
            if sample_start_idx > 0:
                data[:sample_start_idx] = sample[0]
            if sample_end_idx < sequence_length:
                data[sample_end_idx:] = sample[-1]
            data[sample_start_idx:sample_end_idx] = sample
        result[key] = data
    return result

def get_data_stats(data):
    """Compute min/max statistics for normalization."""
    data = data.reshape(-1, data.shape[-1])
    return {'min': np.min(data, axis=0), 'max': np.max(data, axis=0)}

def normalize_data(data, stats):
    """Normalize to [-1, 1]."""
    ndata = (data - stats['min']) / (stats['max'] - stats['min'] + 1e-8)
    return ndata * 2 - 1

class PushTDatasetWithCLIP(Dataset):
    """
    Push-T dataset with CLIP image encoding.
    Loads raw images + proprioceptive state from zarr.
    Images are kept raw for CLIP encoding by the backbone.
    """
    def __init__(self, dataset_path, pred_horizon=16, obs_horizon=2, action_horizon=8):
        # Load zarr dataset
        dataset_root = zarr.open(dataset_path, mode='r')
        train_data = {
            'action': dataset_root['data']['action'][:],
            'state': dataset_root['data']['state'][:],
            'img': dataset_root['data']['img'][:],
        }
        episode_ends = dataset_root['meta']['episode_ends'][:]

        # Create sample indices
        indices = create_sample_indices(
            episode_ends=episode_ends,
            sequence_length=pred_horizon,
            pad_before=obs_horizon - 1,
            pad_after=action_horizon - 1)

        # Normalize action and state data to [-1, 1]
        stats = {}
        normalized_train_data = {}
        for key in ['action', 'state']:
            stats[key] = get_data_stats(train_data[key])
            normalized_train_data[key] = normalize_data(train_data[key], stats[key])
        
        # Keep raw images (uint8 or float) for CLIP encoding
        normalized_train_data['img'] = train_data['img'].astype(np.float32) / 255.0

        self.indices = indices
        self.stats = stats
        self.normalized_train_data = normalized_train_data
        self.pred_horizon = pred_horizon
        self.obs_horizon = obs_horizon
        self.action_horizon = action_horizon

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        buffer_start_idx, buffer_end_idx, sample_start_idx, sample_end_idx = self.indices[idx]

        nsample = sample_sequence(
            train_data=self.normalized_train_data,
            sequence_length=self.pred_horizon,
            buffer_start_idx=buffer_start_idx,
            buffer_end_idx=buffer_end_idx,
            sample_start_idx=sample_start_idx,
            sample_end_idx=sample_end_idx)

        # Return tensors
        return {
            'image': torch.from_numpy(nsample['img']).float(),  # (T, H, W, C)
            'state': torch.from_numpy(nsample['state']).float(),  # (T, state_dim)
            'action': torch.from_numpy(nsample['action']).float(),  # (T, action_dim)
        }

# Load Push-T dataset from zarr
dataset_path = "/Users/ncorrell/Library/CloudStorage/OneDrive-UCB-O365/Desktop/random/liquidnets/transfomer/pusht_cchi_v7_replay.zarr"

full_dataset = PushTDatasetWithCLIP(
    dataset_path=dataset_path,
    pred_horizon=16,
    obs_horizon=2,
    action_horizon=8
)

# Create train/val/test splits
train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset,
    [train_size, val_size, test_size]
)

# Create dataloaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

print(f"✓ Push-T dataset loaded: {len(full_dataset)} samples")
print(f"  Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# Inspect a batch
sample_batch = next(iter(train_loader))
print(f"\nSample batch shapes:")
for k, v in sample_batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}")
print(f"\nNote: Images are raw [0,1] floats. CLIP encoding happens in the backbone.")

## 3. Shared Backbone Architecture

Define a Transformer-based backbone that both diffusion and liquid models will use.
This ensures capacity fairness — only the trajectory generator mechanism differs.

In [ ]:
class SharedBackbone(nn.Module):
    """
    Transformer backbone for Push-T trajectory generation.
    
    Inputs:
    - images: (B, T, H, W, C) — raw images [0, 1]
    - state: (B, T, state_dim) — proprioceptive state (5D for Push-T)
    - clip_model: frozen CLIP model for image encoding
    
    Outputs:
    - context: (B, D_hidden) — aggregated conditioning
    - latent_seq: (B, T, D_hidden) — sequence embeddings
    """
    def __init__(
        self,
        clip_model,
        clip_feat_dim: int = 512,
        state_dim: int = 5,
        hidden_dim: int = 256,
        num_layers: int = 4,
        num_heads: int = 8,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.clip_model = clip_model
        self.hidden_dim = hidden_dim
        self.clip_feat_dim = clip_feat_dim
        
        # Freeze CLIP
        for param in self.clip_model.parameters():
            param.requires_grad = False
        
        # Project CLIP image features to hidden space
        self.image_proj = nn.Linear(clip_feat_dim, hidden_dim)
        # Project proprioceptive state to hidden space (5D for Push-T)
        self.state_proj = nn.Linear(state_dim, hidden_dim)
        
        # Positional encoding for sequences
        self.pos_emb = nn.Embedding(16, hidden_dim)
        
        # Transformer encoder for joint vision-proprioception processing
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Global context aggregation
        self.context_proj = nn.Linear(hidden_dim, hidden_dim)
        
    def encode_images(self, images: torch.Tensor) -> torch.Tensor:
        """
        Encode raw images using frozen CLIP.
        
        Args:
            images: (B, T, H, W, C) in [0, 1]
        
        Returns:
            embeddings: (B, T, clip_feat_dim)
        """
        B, T, H, W, C = images.shape
        
        # Reshape to (B*T, H, W, C)
        images_flat = images.reshape(B * T, H, W, C)
        
        # Convert [0, 1] to CLIP format
        images_for_clip = images_flat.permute(0, 3, 1, 2)  # (B*T, C, H, W)
        
        # Encode with CLIP (returns normalized embeddings)
        with torch.no_grad():
            embeddings = self.clip_model.encode_image(images_for_clip)  # (B*T, clip_feat_dim)
        
        # Ensure embeddings are on same device as input images (handles mock CLIP on CPU)
        embeddings = embeddings.to(images.device)
        
        # Reshape back to (B, T, clip_feat_dim)
        embeddings = embeddings.reshape(B, T, self.clip_feat_dim)
        return embeddings
        
    def forward(self, images: torch.Tensor, state: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            images: (B, T, H, W, C) raw images
            state: (B, T, state_dim) proprioceptive state
        
        Returns:
            context: (B, D_hidden) global conditioning
            latent_seq: (B, T, D_hidden) sequence embeddings
        """
        B, T = images.shape[0], images.shape[1]
        
        # Encode images with CLIP
        img_emb = self.encode_images(images)  # (B, T, clip_feat_dim)
        
        # Project to hidden space
        img_proj = self.image_proj(img_emb)  # (B, T, hidden_dim)
        state_proj = self.state_proj(state)  # (B, T, hidden_dim)
        
        # Combine image and state features
        combined_feat = img_proj + state_proj  # (B, T, hidden_dim)
        
        # Add positional encoding
        pos_ids = torch.arange(T, device=images.device)
        pos_enc = self.pos_emb(pos_ids).unsqueeze(0)  # (1, T, hidden_dim)
        combined_feat = combined_feat + pos_enc
        
        # Transformer encoding
        latent_seq = self.transformer(combined_feat)  # (B, T, hidden_dim)
        
        # Global context: average pooling over time
        context = self.context_proj(latent_seq.mean(dim=1))  # (B, hidden_dim)
        
        return context, latent_seq

# Instantiate shared backbone with frozen CLIP
backbone = SharedBackbone(
    clip_model=clip_model,
    clip_feat_dim=CLIP_FEAT_DIM,
    state_dim=5,  # Push-T has 5D state
    hidden_dim=256,
    num_layers=4
).to(device)
print(f"✓ Shared backbone created (with CLIP image encoding)")
print(f"  Parameters: {sum(p.numel() for p in backbone.parameters()):,}")

# Test forward pass
test_batch = next(iter(train_loader))
test_ctx, test_latent = backbone(
    test_batch['image'].to(device),
    test_batch['state'].to(device)
)
print(f"  Context shape: {test_ctx.shape} | Latent shape: {test_latent.shape}")

## 4. Diffusion Baseline Model

Implement a diffusion-based trajectory generator using the shared backbone for conditioning.

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, embed_dim: int):
        super().__init__()
        self.embed_dim = embed_dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        half_dim = self.embed_dim // 2
        freq = torch.exp(
            -math.log(10000.0) * torch.arange(half_dim, device=t.device).float() / max(half_dim - 1, 1)
        )
        angles = t.float().unsqueeze(1) * freq.unsqueeze(0)
        emb = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)
        if emb.shape[-1] < self.embed_dim:
            emb = F.pad(emb, (0, self.embed_dim - emb.shape[-1]))
        return emb


class DiffusionTrajectoryModel(nn.Module):
    """
    Diffusion-based trajectory generator with a proper DDPM denoiser, sinusoidal
    time embeddings, and an auxiliary MDN head for offline density-style scoring.

    Uses the shared backbone for conditioning on Push-T images + state.
    """
    def __init__(
        self,
        backbone: nn.Module,
        action_dim: int = 2,
        hidden_dim: int = 256,
        seq_length: int = 16,
        num_diffusion_steps: int = 50,
        num_mixtures: int = 5,
        time_embed_dim: int = 64,
    ):
        super().__init__()
        self.backbone = backbone
        self.action_dim = action_dim
        self.hidden_dim = hidden_dim
        self.seq_length = seq_length
        self.num_diffusion_steps = num_diffusion_steps
        self.num_mixtures = num_mixtures
        self.time_embed_dim = time_embed_dim

        backbone_hidden_dim = getattr(backbone, 'hidden_dim', hidden_dim)
        self.context_proj = nn.Identity() if backbone_hidden_dim == hidden_dim else nn.Linear(backbone_hidden_dim, hidden_dim)

        self.register_buffer('betas', torch.linspace(0.0001, 0.02, num_diffusion_steps))
        self.register_buffer('alphas', 1.0 - self.betas)
        self.register_buffer('alphas_cumprod', torch.cumprod(self.alphas, dim=0))
        self.register_buffer(
            'alphas_cumprod_prev',
            torch.cat([torch.ones(1), torch.cumprod(self.alphas, dim=0)[:-1]], dim=0)
        )
        self.register_buffer('sqrt_alphas_cumprod', torch.sqrt(self.alphas_cumprod))
        self.register_buffer('sqrt_one_minus_alphas_cumprod', torch.sqrt(1.0 - self.alphas_cumprod))
        self.register_buffer(
            'posterior_variance',
            self.betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        )
        self.register_buffer(
            'posterior_mean_coef1',
            self.betas * torch.sqrt(self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        )
        self.register_buffer(
            'posterior_mean_coef2',
            (1.0 - self.alphas_cumprod_prev) * torch.sqrt(self.alphas) / (1.0 - self.alphas_cumprod)
        )

        self.time_embed = SinusoidalTimeEmbedding(time_embed_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(time_embed_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        input_dim = action_dim * seq_length + hidden_dim + hidden_dim
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.denoise_net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim * 2),
            nn.SiLU(),
            nn.Linear(hidden_dim * 2, hidden_dim * 2),
            nn.SiLU(),
            nn.Linear(hidden_dim * 2, hidden_dim),
        )
        self.eps_head = nn.Linear(hidden_dim, action_dim * seq_length)

        self.mdn_logits = nn.Linear(hidden_dim, num_mixtures)
        self.mdn_mu = nn.Linear(hidden_dim, num_mixtures * action_dim)
        self.mdn_log_sigma = nn.Linear(hidden_dim, num_mixtures * action_dim)

    def get_noise_schedule(self, t: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        idx = torch.clamp(t, 0, self.num_diffusion_steps - 1)
        alpha_bar_t = self.sqrt_alphas_cumprod[idx]
        sigma_t = self.sqrt_one_minus_alphas_cumprod[idx]
        return alpha_bar_t, sigma_t

    def denoise_step(self, x_t: torch.Tensor, context: torch.Tensor, t: torch.Tensor):
        t_feat = self.time_mlp(self.time_embed(t))
        denoiser_input = torch.cat([x_t, context, t_feat], dim=-1)
        hidden = self.denoise_net(denoiser_input) + self.input_proj(denoiser_input)
        hidden = F.silu(hidden)
        pred_epsilon = self.eps_head(hidden)
        return hidden, pred_epsilon

    def mdn_from_hidden(self, hidden: torch.Tensor, batch_size: int, seq_len: int):
        logits = torch.stack([self.mdn_logits(hidden) for _ in range(seq_len)], dim=1)
        mu = torch.stack([
            self.mdn_mu(hidden).reshape(batch_size, self.num_mixtures, self.action_dim)
            for _ in range(seq_len)
        ], dim=1)
        log_sigma = torch.stack([
            self.mdn_log_sigma(hidden).reshape(batch_size, self.num_mixtures, self.action_dim)
            for _ in range(seq_len)
        ], dim=1)
        return logits, mu, log_sigma

    def forward(
        self,
        images: torch.Tensor,
        state: torch.Tensor,
        action_target: Optional[torch.Tensor] = None,
        return_mdn: bool = False,
        inference: bool = False,
    ) -> torch.Tensor:
        B, T = images.shape[0], images.shape[1]

        context, _ = self.backbone(images, state)
        context = self.context_proj(context)

        if not inference:
            if action_target is None:
                raise ValueError("`action_target` is required during diffusion training/evaluation.")

            target_flat = action_target.reshape(B, -1)
            t = torch.randint(0, self.num_diffusion_steps, (B,), device=images.device)
            alpha_bar_t, sigma_t = self.get_noise_schedule(t)
            epsilon = torch.randn_like(target_flat)
            noisy_actions = alpha_bar_t.view(-1, 1) * target_flat + sigma_t.view(-1, 1) * epsilon

            hidden, pred_epsilon = self.denoise_step(noisy_actions, context, t)
            logits, mu, log_sigma = self.mdn_from_hidden(hidden, B, T)
            ddpm_loss = F.mse_loss(pred_epsilon, epsilon)

            if return_mdn:
                return {
                    'logits': logits,
                    'mu': mu,
                    'log_sigma': log_sigma,
                    'pred_epsilon': pred_epsilon,
                    'target_epsilon': epsilon,
                    'ddpm_loss': ddpm_loss,
                }

            return ddpm_loss

        x_t = torch.randn(B, self.action_dim * self.seq_length, device=images.device)
        for step in range(self.num_diffusion_steps - 1, -1, -1):
            t = torch.full((B,), step, dtype=torch.long, device=images.device)
            _, pred_epsilon = self.denoise_step(x_t, context, t)

            sqrt_alpha_bar = self.sqrt_alphas_cumprod[step]
            sqrt_one_minus_alpha_bar = self.sqrt_one_minus_alphas_cumprod[step]
            pred_x0 = (x_t - sqrt_one_minus_alpha_bar * pred_epsilon) / max(sqrt_alpha_bar.item(), 1e-8)
            pred_x0 = pred_x0.clamp(-1.0, 1.0)

            mean = self.posterior_mean_coef1[step] * pred_x0 + self.posterior_mean_coef2[step] * x_t
            if step > 0:
                noise = torch.randn_like(x_t)
                x_t = mean + torch.sqrt(self.posterior_variance[step].clamp_min(1e-8)) * noise
            else:
                x_t = mean

        return x_t.reshape(B, self.seq_length, self.action_dim).clamp(-1.0, 1.0)

# Instantiate diffusion model
DIFFUSION_PREVIEW_HIDDEN_DIM = 608  # keeps diffusion near 2x the liquid parameter count
diffusion_model = DiffusionTrajectoryModel(
    backbone=backbone,
    action_dim=2,
    hidden_dim=DIFFUSION_PREVIEW_HIDDEN_DIM,
    seq_length=16,
    num_diffusion_steps=50,
    num_mixtures=5,
).to(device)
print(f"✓ Diffusion model created")
print(f"  Diffusion hidden dim: {DIFFUSION_PREVIEW_HIDDEN_DIM}")
print(f"  Diffusion steps: {diffusion_model.num_diffusion_steps}")
print(f"  Total parameters: {sum(p.numel() for p in diffusion_model.parameters()):,}")

# Test forward (training mode)
test_loss = diffusion_model(
    test_batch['image'].to(device),
    test_batch['state'].to(device),
    action_target=test_batch['action'].to(device),
    return_mdn=False,
    inference=False,
)
print(f"  Sample DDPM training loss: {test_loss.item():.4f}")

## 5. Liquid Net Trajectory Model

Implement a liquid neural network (CfC-based) trajectory generator using the same shared backbone.

In [ ]:
class CfCCell(nn.Module):
    """Closed-form Continuous-time (CfC) cell — liquid neural network unit."""
    def __init__(self, input_size: int, hidden_size: int, dropout: float = 0.1):
        super().__init__()
        self.tau = nn.Parameter(torch.ones(hidden_size) * 0.1)
        self.W_gate = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_cand = nn.Linear(input_size + hidden_size, hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, h: torch.Tensor) -> torch.Tensor:
        combined = self.dropout(torch.cat([x, h], dim=-1))
        f = torch.sigmoid(self.W_gate(combined))
        c = torch.tanh(self.W_cand(combined))
        return f * c + (1.0 - f) * h


class LiquidTrajectoryModel(nn.Module):
    """
    Liquid neural network trajectory generator with MDN output.
    Uses shared backbone conditioned on Push-T images + state.
    """
    def __init__(
        self,
        backbone: nn.Module,
        action_dim: int = 2,
        hidden_dim: int = 256,
        seq_length: int = 16,
        num_layers: int = 2,
        num_mixtures: int = 5,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.backbone = backbone
        self.action_dim = action_dim
        self.hidden_dim = hidden_dim
        self.seq_length = seq_length
        self.num_mixtures = num_mixtures

        backbone_hidden_dim = getattr(backbone, 'hidden_dim', hidden_dim)
        self.context_proj = nn.Identity() if backbone_hidden_dim == hidden_dim else nn.Linear(backbone_hidden_dim, hidden_dim)
        self.latent_proj = nn.Identity() if backbone_hidden_dim == hidden_dim else nn.Linear(backbone_hidden_dim, hidden_dim)

        self.cfc_layers = nn.ModuleList([
            CfCCell(hidden_dim, hidden_dim, dropout=dropout)
            for _ in range(num_layers)
        ])

        self.mdn_logits = nn.Linear(hidden_dim, num_mixtures)
        self.mdn_mu = nn.Linear(hidden_dim, num_mixtures * action_dim)
        self.mdn_log_sigma = nn.Linear(hidden_dim, num_mixtures * action_dim)

    def forward(
        self,
        images: torch.Tensor,
        state: torch.Tensor,
        action_target: Optional[torch.Tensor] = None,
        teacher_forcing_ratio: float = 0.0,
        return_mdn: bool = False,
    ) -> torch.Tensor:
        B, T = images.shape[0], images.shape[1]

        context, latent_imgs = self.backbone(images, state)
        context = self.context_proj(context)
        latent_imgs = self.latent_proj(latent_imgs)
        h_states = [context.clone() for _ in self.cfc_layers]

        actions_list = []
        mdn_logits_list, mdn_mu_list, mdn_log_sigma_list = [], [], []

        for t in range(T):
            x_t = latent_imgs[:, t, :]
            for i, layer in enumerate(self.cfc_layers):
                h_states[i] = layer(x_t, h_states[i])
                x_t = h_states[i]

            logits = self.mdn_logits(h_states[-1])
            mu = self.mdn_mu(h_states[-1]).reshape(B, self.num_mixtures, self.action_dim)
            log_sigma = self.mdn_log_sigma(h_states[-1]).reshape(B, self.num_mixtures, self.action_dim)

            mdn_logits_list.append(logits)
            mdn_mu_list.append(mu)
            mdn_log_sigma_list.append(log_sigma)

            if self.training and action_target is not None and torch.rand(1).item() < teacher_forcing_ratio:
                action_t = action_target[:, t, :]
            else:
                mix_idx = torch.multinomial(F.softmax(logits, dim=-1), 1).squeeze(-1)
                mu_t = mu[torch.arange(B, device=mu.device), mix_idx, :]
                sigma_t = torch.exp(log_sigma[torch.arange(B, device=mu.device), mix_idx, :])
                action_t = mu_t + sigma_t * torch.randn_like(mu_t)

            actions_list.append(action_t)

        actions = torch.stack(actions_list, dim=1)

        if return_mdn:
            return {
                'actions': actions,
                'logits': torch.stack(mdn_logits_list, dim=1),
                'mu': torch.stack(mdn_mu_list, dim=1),
                'log_sigma': torch.stack(mdn_log_sigma_list, dim=1),
            }

        if action_target is not None:
            return F.mse_loss(actions, action_target)
        return actions


LIQUID_PREVIEW_HIDDEN_DIM = 291  # ~16 MiB total model size (FP32, incl. shared backbone)

# Instantiate liquid model
liquid_model = LiquidTrajectoryModel(
    backbone=backbone,
    action_dim=2,
    hidden_dim=LIQUID_PREVIEW_HIDDEN_DIM,
    seq_length=16,
    num_layers=2,
    num_mixtures=5,
).to(device)
print(f"✓ Liquid model created")
print(f"  Liquid hidden dim: {LIQUID_PREVIEW_HIDDEN_DIM}")
print(f"  Total parameters: {sum(p.numel() for p in liquid_model.parameters()):,}")

# Test forward (training mode)
test_loss = liquid_model(
    test_batch['image'].to(device),
    test_batch['state'].to(device),
    action_target=test_batch['action'].to(device),
)
print(f"  Sample training loss: {test_loss.item():.4f}")

## 6. Training Pipeline with Frozen Encoders

Train both models using the same loss, optimizer, and schedule.
Keep CLIP encoders and shared backbone frozen during training.

In [ ]:
# ---- Hyperparameters for fair half-parameter comparison ----

import copy



D_MODEL = 256

LIQUID_HIDDEN_SIZE = 291      # ~16 MiB total model (FP32, incl. model-specific backbone)

DIFFUSION_HIDDEN_SIZE = 608   # ~2x the liquid parameter count with the improved denoiser

NUM_LAYERS = 2

NUM_MIXTURES = 5

NUM_DIFFUSION_STEPS = 50

DROPOUT = 0.1

LR = 1e-4

WEIGHT_DECAY = 1e-4

NUM_EPOCHS = 120

WARMUP_EPOCHS = 5

MIN_DELTA = 1e-5

PATIENCE = 999

DATASET_NAME = "pusht"

EXPERIMENT_TAG = "fair_halfparam_deterministic_clip_120epochs"

TRAIN_LIQUID = True

TRAIN_DIFFUSION = True

DIFFUSION_NLL_WEIGHT = 0.25

DIFFUSION_DDPM_WEIGHT = 1.0

FREEZE_MODEL_BACKBONES = False



# Paths / artifact infrastructure

CHECKPOINT_DIR = "checkpoints"

ARTIFACT_DIR = "artifacts"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

os.makedirs(ARTIFACT_DIR, exist_ok=True)



LIQUID_CKPT_PATH = f"{CHECKPOINT_DIR}/liquid_jepa_{DATASET_NAME}_{EXPERIMENT_TAG}_best.pt"

DIFFUSION_CKPT_PATH = f"{CHECKPOINT_DIR}/diffusion_jepa_{DATASET_NAME}_{EXPERIMENT_TAG}_best.pt"

TRAINING_CACHE_PATH = f"{ARTIFACT_DIR}/training_history_{DATASET_NAME}_{EXPERIMENT_TAG}.json"

EVAL_CACHE_PATH = f"{ARTIFACT_DIR}/eval_results_{DATASET_NAME}_{EXPERIMENT_TAG}.json"



def save_json(path, payload):

    with open(path, "w") as f:

        json.dump(payload, f, indent=2)



def load_json(path):

    with open(path, "r") as f:

        return json.load(f)



# MDN negative log-likelihood loss (from liquid_pusht)

def mdn_nll_loss(logits, mu, log_sigma, target):

    """Compute MDN NLL loss for diagonal Gaussian mixture model."""

    target = target.unsqueeze(2)  # (B, T, 1, action_dim)

    z = (target - mu) * torch.exp(-log_sigma)

    log_prob = -0.5 * (z * z + 2.0 * log_sigma + math.log(2.0 * math.pi)).sum(dim=-1)  # (B, T, K)

    log_mix = torch.log_softmax(logits, dim=-1) + log_prob  # (B, T, K)

    nll = -torch.logsumexp(log_mix, dim=-1)  # (B, T)

    return nll.mean()



print("MDN NLL loss function defined.")



# Create independent backbone copies so each model gets the same initialization

# but is trained independently for a fair end-to-end comparison.

liquid_backbone = copy.deepcopy(backbone).to(device)

diffusion_backbone = copy.deepcopy(backbone).to(device)



if FREEZE_MODEL_BACKBONES:

    for module in [liquid_backbone, diffusion_backbone]:

        module.eval()

        for param in module.parameters():

            param.requires_grad = False



# Initialize models

liquid_model = LiquidTrajectoryModel(

    backbone=liquid_backbone,

    action_dim=2,

    hidden_dim=LIQUID_HIDDEN_SIZE,

    seq_length=16,

    num_layers=NUM_LAYERS,

    num_mixtures=NUM_MIXTURES,

    dropout=DROPOUT,

).to(device)



diffusion_model = DiffusionTrajectoryModel(

    backbone=diffusion_backbone,

    action_dim=2,

    hidden_dim=DIFFUSION_HIDDEN_SIZE,

    seq_length=16,

    num_diffusion_steps=NUM_DIFFUSION_STEPS,

    num_mixtures=NUM_MIXTURES,

).to(device)



liquid_params = sum(p.numel() for p in liquid_model.parameters())

diffusion_params = sum(p.numel() for p in diffusion_model.parameters())

backbone_params = sum(p.numel() for p in liquid_backbone.parameters())

parameter_ratio = liquid_params / max(diffusion_params, 1)



# Approximate model size (MiB) assuming FP32 weights only

liquid_size_mib = liquid_params * 4 / (1024 ** 2)

diffusion_size_mib = diffusion_params * 4 / (1024 ** 2)



print(f"\n{'='*60}")

print("Architecture Summary")

print(f"{'='*60}")

print(f"Experiment tag:            {EXPERIMENT_TAG}")

print(f"Shared backbone template:  {backbone_params:,} params")

print(f"Liquid Net:                {liquid_params:,} params  (~{liquid_size_mib:.2f} MiB)")

print(f"  └─ liquid hidden dim:    {LIQUID_HIDDEN_SIZE}")

print(f"Diffusion Net:             {diffusion_params:,} params  (~{diffusion_size_mib:.2f} MiB)")

print(f"  └─ diffusion hidden:     {DIFFUSION_HIDDEN_SIZE}")

print(f"Liquid / Diffusion params: {parameter_ratio:.3f}x")

print(f"Diffusion Steps:           {NUM_DIFFUSION_STEPS}")

print(f"Freeze model backbones:    {FREEZE_MODEL_BACKBONES}")

print(f"{'='*60}\n")



# Optimizers and schedulers

liquid_optimizer = torch.optim.AdamW(

    [p for p in liquid_model.parameters() if p.requires_grad],

    lr=LR,

    weight_decay=WEIGHT_DECAY,

)

diffusion_optimizer = torch.optim.AdamW(

    [p for p in diffusion_model.parameters() if p.requires_grad],

    lr=LR,

    weight_decay=WEIGHT_DECAY,

)



# Warmup + Cosine annealing schedule (from liquid_pusht)

def create_scheduler(optimizer, num_epochs, warmup_epochs):

    warmup = torch.optim.lr_scheduler.LinearLR(

        optimizer,

        start_factor=0.4,

        end_factor=1.0,

        total_iters=warmup_epochs,

    )

    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(

        optimizer,

        T_max=max(1, num_epochs - warmup_epochs),

        eta_min=3e-7,

    )

    return torch.optim.lr_scheduler.SequentialLR(

        optimizer,

        schedulers=[warmup, cosine],

        milestones=[warmup_epochs],

    )



liquid_scheduler = create_scheduler(liquid_optimizer, NUM_EPOCHS, WARMUP_EPOCHS)

diffusion_scheduler = create_scheduler(diffusion_optimizer, NUM_EPOCHS, WARMUP_EPOCHS)



print(f"LR schedule: {WARMUP_EPOCHS}-epoch warmup from {LR*0.4:.2e} to {LR:.2e}, then cosine decay")

print(f"Loss: MDN NLL with K={NUM_MIXTURES} mixtures")

print(f"Diffusion objective: {DIFFUSION_NLL_WEIGHT:.2f} * MDN NLL + {DIFFUSION_DDPM_WEIGHT:.2f} * DDPM noise MSE")

print(f"Train liquid: {TRAIN_LIQUID} | Train diffusion: {TRAIN_DIFFUSION}")

print(f"Early stopping: patience={PATIENCE}, min_delta={MIN_DELTA:.1e}")

print(f"Liquid checkpoint: {LIQUID_CKPT_PATH}")

print(f"Diffusion checkpoint: {DIFFUSION_CKPT_PATH}")

print(f"Training cache: {TRAINING_CACHE_PATH}")

print(f"Eval cache: {EVAL_CACHE_PATH}\n")



# Training state

best_liquid_loss = float('inf')

best_diffusion_loss = float('inf')

liquid_no_improve = 0

diffusion_no_improve = 0


## 7. Fair Offline Evaluation Protocol

This notebook compares **trajectory prediction quality**, not closed-loop control success.

### Fairness controls
- Same dataset split, normalization, and test loader
- Same observation inputs: image + state
- Same prediction horizon and action representation
- Same frozen backbone design and conditioning pathway
- Same sampling budget $K$ for stochastic evaluation
- Fixed diffusion step count during reverse sampling

### Core reported metrics
- **NLL**: exact MDN negative log-likelihood for liquid; diffusion reports a proxy MDN score at $t=0$
- **MSE**: deterministic trajectory error
- **Best-of-$K$ MSE**: for each example, pick the minimum trajectory MSE across $K$ samples
- **Diversity**: average pairwise trajectory distance across samples
- **Smoothness**: jerk penalty, using the second finite difference
- **Latency**: wall-clock batch time for deterministic and stochastic sampling

### Exact definitions
For a sampled set of trajectories $\{a^{(k)}\}_{k=1}^K$ and ground-truth trajectory $a^{gt}$:

- Best-of-$K$ MSE:
  $$
  \frac{1}{N}\sum_{n=1}^N \min_{k \in \{1,\dots,K\}} \left[\frac{1}{TH}\lVert a_n^{(k)} - a_n^{gt} \rVert_2^2\right]
  $$
- Diversity:
  $$
  \frac{1}{N}\sum_{n=1}^N \frac{1}{K(K-1)}\sum_{i \ne j} d\left(a_n^{(i)}, a_n^{(j)}\right)
  $$
  where $d(\cdot,\cdot)$ is the trajectory RMS distance.
- Smoothness / jerk:
  $$
  \frac{1}{N}\sum_{n=1}^N \frac{1}{T-2}\sum_{t=2}^{T-1} \left\lVert a_{t+1} - 2a_t + a_{t-1} \right\rVert_2^2
  $$

This makes the comparison reproducible and hard to attack in review.

In [ ]:
  # Training loop with selective repair for the updated diffusion model
import time

existing_history = load_json(TRAINING_CACHE_PATH) if os.path.exists(TRAINING_CACHE_PATH) else {}
liquid_train_baseline = float(existing_history.get('liquid_train_nll', [-1.0])[-1]) if existing_history.get('liquid_train_nll') else -1.0
liquid_val_baseline = float(existing_history.get('liquid_val_nll', [-1.0])[-1]) if existing_history.get('liquid_val_nll') else -1.0

training_history = {
    "dataset": DATASET_NAME,
    "num_epochs_config": int(NUM_EPOCHS),
    "num_diffusion_steps": int(NUM_DIFFUSION_STEPS),
    "train_liquid": bool(TRAIN_LIQUID),
    "train_diffusion": bool(TRAIN_DIFFUSION),
    "diffusion_nll_weight": float(DIFFUSION_NLL_WEIGHT),
    "diffusion_ddpm_weight": float(DIFFUSION_DDPM_WEIGHT),
    "liquid_checkpoint": LIQUID_CKPT_PATH,
    "diffusion_checkpoint": DIFFUSION_CKPT_PATH,
    "epochs": [],
    "teacher_forcing_ratio": [],
    "free_running_weight": [],
    "liquid_train_nll": [],
    "liquid_val_nll": [],
    "diffusion_train_nll": [],
    "diffusion_val_nll": [],
    "diffusion_train_ddpm": [],
    "diffusion_val_ddpm": [],
    "diffusion_train_total": [],
    "diffusion_val_total": [],
    "liquid_lr": [],
    "diffusion_lr": [],
    "epoch_time_sec": []
}

if (not TRAIN_LIQUID) and os.path.exists(LIQUID_CKPT_PATH):
    liquid_model.load_state_dict(torch.load(LIQUID_CKPT_PATH, map_location=device))
    liquid_model.eval()
    print(f"Loaded frozen liquid checkpoint from {LIQUID_CKPT_PATH}")

print("\nStarting training / repair run...")
for epoch in range(NUM_EPOCHS):
    epoch_start = time.perf_counter()

    progress = epoch / max(1, NUM_EPOCHS - 1)
    teacher_forcing_ratio = max(0.15, 1.0 - 0.90 * progress)
    free_w = min(0.75, 0.20 + 0.65 * progress)
    tf_w = 1.0 - free_w

    # ---- Train Liquid Net ----
    if TRAIN_LIQUID:
        liquid_model.train()
        liquid_train_loss = 0.0

        with tqdm(train_loader, desc=f"[Liquid] Epoch {epoch}", leave=False) as pbar:
            for batch in pbar:
                images = batch['image'].to(device)
                state = batch['state'].to(device)
                action_target = batch['action'].to(device)

                liquid_optimizer.zero_grad()

                out_tf = liquid_model(
                    images, state,
                    action_target=action_target,
                    teacher_forcing_ratio=teacher_forcing_ratio,
                    return_mdn=True,
                )
                loss_tf = mdn_nll_loss(out_tf['logits'], out_tf['mu'], out_tf['log_sigma'], action_target)

                out_free = liquid_model(
                    images, state,
                    action_target=None,
                    teacher_forcing_ratio=0.0,
                    return_mdn=True,
                )
                loss_free = mdn_nll_loss(out_free['logits'], out_free['mu'], out_free['log_sigma'], action_target)

                loss = tf_w * loss_tf + free_w * loss_free
                loss.backward()
                torch.nn.utils.clip_grad_norm_(liquid_model.parameters(), 1.0)
                liquid_optimizer.step()

                liquid_train_loss += loss.item()
                pbar.set_postfix(loss=f"{loss.item():.5f}")

        avg_liquid_train = liquid_train_loss / len(train_loader)
    else:
        avg_liquid_train = liquid_train_baseline

    # ---- Train Diffusion ----
    if TRAIN_DIFFUSION:
        diffusion_model.train()
        diffusion_train_nll = 0.0
        diffusion_train_ddpm = 0.0
        diffusion_train_total = 0.0

        with tqdm(train_loader, desc=f"[Diffusion] Epoch {epoch}", leave=False) as pbar:
            for batch in pbar:
                images = batch['image'].to(device)
                state = batch['state'].to(device)
                action_target = batch['action'].to(device)

                diffusion_optimizer.zero_grad()
                out = diffusion_model(
                    images, state,
                    action_target=action_target,
                    return_mdn=True,
                    inference=False,
                )
                loss_nll = mdn_nll_loss(out['logits'], out['mu'], out['log_sigma'], action_target)
                loss_ddpm = F.mse_loss(out['pred_epsilon'], out['target_epsilon'])
                loss = DIFFUSION_NLL_WEIGHT * loss_nll + DIFFUSION_DDPM_WEIGHT * loss_ddpm

                loss.backward()
                torch.nn.utils.clip_grad_norm_(diffusion_model.parameters(), 1.0)
                diffusion_optimizer.step()

                diffusion_train_nll += loss_nll.item()
                diffusion_train_ddpm += loss_ddpm.item()
                diffusion_train_total += loss.item()
                pbar.set_postfix(total=f"{loss.item():.5f}", nll=f"{loss_nll.item():.5f}", ddpm=f"{loss_ddpm.item():.5f}")

        avg_diffusion_train_nll = diffusion_train_nll / len(train_loader)
        avg_diffusion_train_ddpm = diffusion_train_ddpm / len(train_loader)
        avg_diffusion_train = diffusion_train_total / len(train_loader)
    else:
        avg_diffusion_train_nll = 0.0
        avg_diffusion_train_ddpm = 0.0
        avg_diffusion_train = 0.0

    # ---- Validate ----
    liquid_model.eval()
    diffusion_model.eval()
    liquid_val_nll = 0.0
    diffusion_val_nll = 0.0
    diffusion_val_ddpm = 0.0
    diffusion_val_total = 0.0

    with torch.no_grad():
        for batch in val_loader:
            images = batch['image'].to(device)
            state = batch['state'].to(device)
            action_target = batch['action'].to(device)

            out_liquid = liquid_model(images, state, teacher_forcing_ratio=0.0, return_mdn=True)
            loss_liquid = mdn_nll_loss(out_liquid['logits'], out_liquid['mu'], out_liquid['log_sigma'], action_target)
            liquid_val_nll += loss_liquid.item()

            out_diffusion = diffusion_model(
                images, state,
                action_target=action_target,
                return_mdn=True,
                inference=False,
            )
            loss_diffusion_nll = mdn_nll_loss(out_diffusion['logits'], out_diffusion['mu'], out_diffusion['log_sigma'], action_target)
            loss_diffusion_ddpm = F.mse_loss(out_diffusion['pred_epsilon'], out_diffusion['target_epsilon'])
            loss_diffusion_total = DIFFUSION_NLL_WEIGHT * loss_diffusion_nll + DIFFUSION_DDPM_WEIGHT * loss_diffusion_ddpm

            diffusion_val_nll += loss_diffusion_nll.item()
            diffusion_val_ddpm += loss_diffusion_ddpm.item()
            diffusion_val_total += loss_diffusion_total.item()

    avg_liquid_val = liquid_val_nll / len(val_loader)
    avg_diffusion_val_nll = diffusion_val_nll / len(val_loader)
    avg_diffusion_val_ddpm = diffusion_val_ddpm / len(val_loader)
    avg_diffusion_val = diffusion_val_total / len(val_loader)

    if TRAIN_LIQUID:
        liquid_scheduler.step()
    if TRAIN_DIFFUSION:
        diffusion_scheduler.step()

    if TRAIN_LIQUID:
        if avg_liquid_val < (best_liquid_loss - MIN_DELTA):
            best_liquid_loss = avg_liquid_val
            liquid_no_improve = 0
            torch.save(liquid_model.state_dict(), LIQUID_CKPT_PATH)
            liquid_tag = "  [best]"
        else:
            liquid_no_improve += 1
            liquid_tag = ""
    else:
        liquid_tag = "  [frozen]"
        best_liquid_loss = avg_liquid_val if best_liquid_loss == float('inf') else best_liquid_loss

    if TRAIN_DIFFUSION:
        if avg_diffusion_val < (best_diffusion_loss - MIN_DELTA):
            best_diffusion_loss = avg_diffusion_val
            diffusion_no_improve = 0
            torch.save(diffusion_model.state_dict(), DIFFUSION_CKPT_PATH)
            diffusion_tag = "  [best]"
        else:
            diffusion_no_improve += 1
            diffusion_tag = ""
    else:
        diffusion_tag = "  [frozen]"
        best_diffusion_loss = avg_diffusion_val if best_diffusion_loss == float('inf') else best_diffusion_loss

    epoch_time = time.perf_counter() - epoch_start
    training_history["epochs"].append(int(epoch))
    training_history["teacher_forcing_ratio"].append(float(teacher_forcing_ratio))
    training_history["free_running_weight"].append(float(free_w))
    training_history["liquid_train_nll"].append(float(avg_liquid_train))
    training_history["liquid_val_nll"].append(float(avg_liquid_val))
    training_history["diffusion_train_nll"].append(float(avg_diffusion_train_nll))
    training_history["diffusion_val_nll"].append(float(avg_diffusion_val_nll))
    training_history["diffusion_train_ddpm"].append(float(avg_diffusion_train_ddpm))
    training_history["diffusion_val_ddpm"].append(float(avg_diffusion_val_ddpm))
    training_history["diffusion_train_total"].append(float(avg_diffusion_train))
    training_history["diffusion_val_total"].append(float(avg_diffusion_val))
    training_history["liquid_lr"].append(float(liquid_optimizer.param_groups[0]["lr"]))
    training_history["diffusion_lr"].append(float(diffusion_optimizer.param_groups[0]["lr"]))
    training_history["epoch_time_sec"].append(float(epoch_time))

    save_json(TRAINING_CACHE_PATH, training_history)

    print(f"Epoch {epoch} | TF: {teacher_forcing_ratio:.3f} | Free: {free_w:.3f} | Time: {epoch_time:.1f}s")
    print(f"  [Liquid]    Train NLL: {avg_liquid_train:.5f} | Val NLL: {avg_liquid_val:.5f}{liquid_tag}")
    print(f"  [Diffusion] Train total: {avg_diffusion_train:.5f} | Val total: {avg_diffusion_val:.5f}{diffusion_tag}")
    print(f"               Train NLL: {avg_diffusion_train_nll:.5f} | Val NLL: {avg_diffusion_val_nll:.5f}")
    print(f"               Train DDPM: {avg_diffusion_train_ddpm:.5f} | Val DDPM: {avg_diffusion_val_ddpm:.5f}")

    if TRAIN_DIFFUSION and diffusion_no_improve >= PATIENCE:
        print(f"Early stopping diffusion repair: no improvement for {PATIENCE} epochs.")
        break

training_history["best_liquid_val_nll"] = float(best_liquid_loss if best_liquid_loss != float('inf') else liquid_val_baseline)
training_history["best_diffusion_val_total"] = float(best_diffusion_loss)
training_history["epochs_completed"] = len(training_history["epochs"])
save_json(TRAINING_CACHE_PATH, training_history)

print("\n✓ Training completed")
print(f"  Best Liquid Val NLL: {training_history['best_liquid_val_nll']:.5f}")
print(f"  Best Diffusion Val Total: {best_diffusion_loss:.5f}")
print(f"  Training history saved to: {TRAINING_CACHE_PATH}")

In [ ]:
import time

K_VALUES = [1, 2, 5, 10]
NUM_EVAL_SAMPLES = max(K_VALUES)
NUM_QUAL_EXAMPLES = 3
FORCE_REEVAL = True  # Set True to ignore cached eval results
ARTIFACT_STEM = f"{DATASET_NAME}_{EXPERIMENT_TAG}"

# Load best checkpoints
liquid_model.load_state_dict(torch.load(LIQUID_CKPT_PATH, map_location=device))
diffusion_model.load_state_dict(torch.load(DIFFUSION_CKPT_PATH, map_location=device))
liquid_model.eval()
diffusion_model.eval()


def compute_jerk(trajs: torch.Tensor) -> torch.Tensor:
    """Mean squared second finite difference along time."""
    if trajs.shape[-2] < 3:
        return torch.zeros(trajs.shape[:-2], device=trajs.device)
    jerk = trajs[..., 2:, :] - 2.0 * trajs[..., 1:-1, :] + trajs[..., :-2, :]
    return (jerk ** 2).mean(dim=(-1, -2))


def compute_deterministic_metrics(pred: torch.Tensor, target: torch.Tensor):
    per_step_mse = ((pred - target) ** 2).mean(dim=-1)  # (B, T)
    return {
        'mse': per_step_mse.mean().item(),
        'smoothness': compute_jerk(pred).mean().item(),
        'per_horizon_mse': per_step_mse.mean(dim=0).detach().cpu().tolist(),
    }


def compute_sample_metrics(samples: torch.Tensor, target: torch.Tensor):
    """
    Sample metrics for a batch.

    samples: (B, K, T, A)
    target:  (B, T, A)
    """
    per_step_mse = ((samples - target.unsqueeze(1)) ** 2).mean(dim=-1)  # (B, K, T)
    traj_mse = per_step_mse.mean(dim=-1)  # (B, K)
    best_idx = traj_mse.argmin(dim=1)
    batch_idx = torch.arange(samples.shape[0], device=samples.device)
    best_traj = samples[batch_idx, best_idx]
    best_step_mse = per_step_mse[batch_idx, best_idx]

    if samples.shape[1] > 1:
        diffs = samples.unsqueeze(2) - samples.unsqueeze(1)
        pairwise_rms = torch.sqrt((diffs ** 2).mean(dim=(-1, -2)) + 1e-12)
        tri_upper = torch.triu_indices(samples.shape[1], samples.shape[1], offset=1, device=samples.device)
        per_example_diversity = pairwise_rms[:, tri_upper[0], tri_upper[1]].mean(dim=1)
        diversity = per_example_diversity.mean().item()
    else:
        per_example_diversity = torch.zeros(samples.shape[0], device=samples.device)
        diversity = 0.0

    return {
        'sample_mean_mse': traj_mse.mean().item(),
        'best_of_k_mse': traj_mse.min(dim=1).values.mean().item(),
        'diversity_l2': diversity,
        'smoothness': compute_jerk(samples).mean().item(),
        'per_horizon_mean_mse': per_step_mse.mean(dim=(0, 1)).detach().cpu().tolist(),
        'per_horizon_best_of_k_mse': best_step_mse.mean(dim=0).detach().cpu().tolist(),
        'per_example_best_of_k_mse': traj_mse.min(dim=1).values.detach().cpu().tolist(),
        'per_example_diversity': per_example_diversity.detach().cpu().tolist(),
    }


def liquid_sample_trajectories(model, images, state, num_samples):
    samples = []
    with torch.no_grad():
        for _ in range(num_samples):
            out = model(images, state, teacher_forcing_ratio=0.0, return_mdn=True)
            samples.append(out['actions'])
    return torch.stack(samples, dim=1)


def liquid_mean_trajectory(model, images, state):
    with torch.no_grad():
        out = model(images, state, teacher_forcing_ratio=0.0, return_mdn=True)
        weights = F.softmax(out['logits'], dim=-1)
        pred_mean = (weights.unsqueeze(-1) * out['mu']).sum(dim=2)
    return pred_mean, out


def diffusion_sample_trajectories(model, images, state, num_samples):
    samples = []
    with torch.no_grad():
        for _ in range(num_samples):
            samples.append(model(images, state, inference=True))
    return torch.stack(samples, dim=1)


def diffusion_proxy_density(model, images, state, action_target):
    """Proxy density score for diffusion via the MDN head at t=0."""
    B, T = images.shape[0], images.shape[1]
    context, _ = model.backbone(images, state)
    context = model.context_proj(context)
    target_flat = action_target.reshape(B, -1)

    t_zero = torch.zeros(B, dtype=torch.long, device=images.device)
    alpha_t, sigma_t = model.get_noise_schedule(t_zero)
    noisy = alpha_t.view(-1, 1) * target_flat + sigma_t.view(-1, 1) * torch.randn_like(target_flat)
    hidden, _ = model.denoise_step(noisy, context, t_zero)
    logits, mu, log_sigma = model.mdn_from_hidden(hidden, B, T)
    weights = F.softmax(logits, dim=-1)
    pred_mean = (weights.unsqueeze(-1) * mu).sum(dim=2)
    return {
        'logits': logits,
        'mu': mu,
        'log_sigma': log_sigma,
        'pred_mean': pred_mean,
    }


def init_k_accumulators():
    return {
        str(k): {
            'sample_mean_mse': 0.0,
            'best_of_k_mse': 0.0,
            'diversity_l2': 0.0,
            'smoothness': 0.0,
            'per_horizon_mean_mse': None,
            'per_horizon_best_of_k_mse': None,
            'per_example_best_of_k_mse': [],
            'per_example_diversity': [],
        }
        for k in K_VALUES
    }


def update_k_accumulators(accumulators, samples, target):
    for k in K_VALUES:
        metrics = compute_sample_metrics(samples[:, :k], target)
        bucket = accumulators[str(k)]
        bucket['sample_mean_mse'] += metrics['sample_mean_mse']
        bucket['best_of_k_mse'] += metrics['best_of_k_mse']
        bucket['diversity_l2'] += metrics['diversity_l2']
        bucket['smoothness'] += metrics['smoothness']
        vec_mean = np.asarray(metrics['per_horizon_mean_mse'])
        vec_best = np.asarray(metrics['per_horizon_best_of_k_mse'])
        bucket['per_horizon_mean_mse'] = vec_mean if bucket['per_horizon_mean_mse'] is None else bucket['per_horizon_mean_mse'] + vec_mean
        bucket['per_horizon_best_of_k_mse'] = vec_best if bucket['per_horizon_best_of_k_mse'] is None else bucket['per_horizon_best_of_k_mse'] + vec_best
        bucket['per_example_best_of_k_mse'].extend(metrics['per_example_best_of_k_mse'])
        bucket['per_example_diversity'].extend(metrics['per_example_diversity'])


def finalize_k_accumulators(accumulators, n_batches):
    finalized = {}
    for key, bucket in accumulators.items():
        finalized[key] = {
            'sample_mean_mse': bucket['sample_mean_mse'] / n_batches,
            'best_of_k_mse': bucket['best_of_k_mse'] / n_batches,
            'diversity_l2': bucket['diversity_l2'] / n_batches,
            'smoothness': bucket['smoothness'] / n_batches,
            'per_horizon_mean_mse': (bucket['per_horizon_mean_mse'] / n_batches).tolist(),
            'per_horizon_best_of_k_mse': (bucket['per_horizon_best_of_k_mse'] / n_batches).tolist(),
            'per_example_best_of_k_mse': bucket['per_example_best_of_k_mse'],
            'per_example_diversity': bucket['per_example_diversity'],
        }
    return finalized


def evaluate_liquid(model, test_loader, device, num_samples):
    exact_nll_total = 0.0
    deterministic_mse_total = 0.0
    deterministic_smoothness_total = 0.0
    deterministic_horizon_total = None
    single_pass_times = []
    k_sample_times = []
    k_metrics = init_k_accumulators()
    qualitative = None

    print("\nEvaluating Liquid Net...")
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="[Liquid] Eval", leave=False):
            images = batch['image'].to(device)
            state = batch['state'].to(device)
            action_target = batch['action'].to(device)

            t_start = time.perf_counter()
            pred_mean, out = liquid_mean_trajectory(model, images, state)
            single_pass_times.append(time.perf_counter() - t_start)

            deterministic_metrics = compute_deterministic_metrics(pred_mean, action_target)
            exact_nll_total += mdn_nll_loss(out['logits'], out['mu'], out['log_sigma'], action_target).item()
            deterministic_mse_total += deterministic_metrics['mse']
            deterministic_smoothness_total += deterministic_metrics['smoothness']
            horizon_vec = np.asarray(deterministic_metrics['per_horizon_mse'])
            deterministic_horizon_total = horizon_vec if deterministic_horizon_total is None else deterministic_horizon_total + horizon_vec

            t_start = time.perf_counter()
            samples = liquid_sample_trajectories(model, images, state, num_samples)
            k_sample_times.append(time.perf_counter() - t_start)
            update_k_accumulators(k_metrics, samples, action_target)

            if qualitative is None:
                qualitative = {
                    'target': action_target[:NUM_QUAL_EXAMPLES].detach().cpu().tolist(),
                    'deterministic': pred_mean[:NUM_QUAL_EXAMPLES].detach().cpu().tolist(),
                    'samples': samples[:NUM_QUAL_EXAMPLES, :10].detach().cpu().tolist(),
                }

    n = len(test_loader)
    finalized_k = finalize_k_accumulators(k_metrics, n)
    results = {
        'model_name': 'Liquid + MDN',
        'exact_nll': exact_nll_total / n,
        'single_pass_ms': float(np.mean(single_pass_times) * 1000),
        'k_sample_ms': float(np.mean(k_sample_times) * 1000),
        'deterministic': {
            'mse': deterministic_mse_total / n,
            'best_of_k_mse': deterministic_mse_total / n,
            'diversity_l2': 0.0,
            'smoothness': deterministic_smoothness_total / n,
            'per_horizon_mse': (deterministic_horizon_total / n).tolist(),
        },
        'k_metrics': finalized_k,
        'qualitative': qualitative,
        'num_eval_samples': int(num_samples),
        'density_metric': 'exact MDN NLL',
    }
    k10 = results['k_metrics']['10']
    print(f"  Exact NLL:            {results['exact_nll']:.5f}")
    print(f"  Deterministic MSE:    {results['deterministic']['mse']:.6f}")
    print(f"  Best-of-10 MSE:       {k10['best_of_k_mse']:.6f}")
    print(f"  Diversity @10:        {k10['diversity_l2']:.6f}")
    print(f"  Smoothness @10:       {k10['smoothness']:.6f}")
    print(f"  Time (1 sample):      {results['single_pass_ms']:.3f} ms/batch")
    print(f"  Time (10 samples):    {results['k_sample_ms']:.3f} ms/batch")
    return results


def evaluate_diffusion(model, test_loader, device, num_samples):
    proxy_nll_total = 0.0
    proxy_mse_total = 0.0
    proxy_smoothness_total = 0.0
    proxy_horizon_total = None
    proxy_pass_times = []
    k_sample_times = []
    k_metrics = init_k_accumulators()
    qualitative = None

    print("\nEvaluating Diffusion...")
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="[Diffusion] Eval", leave=False):
            images = batch['image'].to(device)
            state = batch['state'].to(device)
            action_target = batch['action'].to(device)

            t_start = time.perf_counter()
            proxy_out = diffusion_proxy_density(model, images, state, action_target)
            proxy_pass_times.append(time.perf_counter() - t_start)

            proxy_metrics = compute_deterministic_metrics(proxy_out['pred_mean'], action_target)
            proxy_nll_total += mdn_nll_loss(proxy_out['logits'], proxy_out['mu'], proxy_out['log_sigma'], action_target).item()
            proxy_mse_total += proxy_metrics['mse']
            proxy_smoothness_total += proxy_metrics['smoothness']
            horizon_vec = np.asarray(proxy_metrics['per_horizon_mse'])
            proxy_horizon_total = horizon_vec if proxy_horizon_total is None else proxy_horizon_total + horizon_vec

            t_start = time.perf_counter()
            samples = diffusion_sample_trajectories(model, images, state, num_samples)
            k_sample_times.append(time.perf_counter() - t_start)
            update_k_accumulators(k_metrics, samples, action_target)

            if qualitative is None:
                qualitative = {
                    'target': action_target[:NUM_QUAL_EXAMPLES].detach().cpu().tolist(),
                    'proxy_mean': proxy_out['pred_mean'][:NUM_QUAL_EXAMPLES].detach().cpu().tolist(),
                    'samples': samples[:NUM_QUAL_EXAMPLES, :10].detach().cpu().tolist(),
                }

    n = len(test_loader)
    finalized_k = finalize_k_accumulators(k_metrics, n)
    results = {
        'model_name': 'Diffusion',
        'proxy_nll': proxy_nll_total / n,
        'proxy_pass_ms': float(np.mean(proxy_pass_times) * 1000),
        'k_sample_ms': float(np.mean(k_sample_times) * 1000),
        'proxy_deterministic': {
            'mse': proxy_mse_total / n,
            'smoothness': proxy_smoothness_total / n,
            'per_horizon_mse': (proxy_horizon_total / n).tolist(),
        },
        'k_metrics': finalized_k,
        'qualitative': qualitative,
        'num_eval_samples': int(num_samples),
        'density_metric': 'proxy MDN NLL at t=0',
    }
    k10 = results['k_metrics']['10']
    print(f"  Proxy NLL:            {results['proxy_nll']:.5f}")
    print(f"  Proxy mean MSE:       {results['proxy_deterministic']['mse']:.6f}")
    print(f"  Best-of-10 MSE:       {k10['best_of_k_mse']:.6f}")
    print(f"  Diversity @10:        {k10['diversity_l2']:.6f}")
    print(f"  Smoothness @10:       {k10['smoothness']:.6f}")
    print(f"  Time (proxy pass):    {results['proxy_pass_ms']:.3f} ms/batch")
    print(f"  Time (10 samples):    {results['k_sample_ms']:.3f} ms/batch")
    return results


# ── Cache check ──────────────────────────────────────────────────────────────
use_cache = False
cached = None
if (not FORCE_REEVAL) and os.path.exists(EVAL_CACHE_PATH):
    cached = load_json(EVAL_CACHE_PATH)
    cache_steps = int(cached.get('num_diffusion_steps', -1))
    cache_epochs = int(cached.get('epochs_completed', -1))
    cache_samples = int(cached.get('num_eval_samples', -1))
    target_epochs = -1
    if os.path.exists(TRAINING_CACHE_PATH):
        try:
            target_epochs = int(load_json(TRAINING_CACHE_PATH).get('epochs_completed', -1))
        except Exception:
            pass
    if (
        cached.get('dataset') == DATASET_NAME
        and cached.get('experiment_tag') == EXPERIMENT_TAG
        and cache_steps == int(NUM_DIFFUSION_STEPS)
        and cache_samples == int(NUM_EVAL_SAMPLES)
        and (target_epochs < 0 or cache_epochs == target_epochs)
        and 'core_table' in cached
        and 'k_values' in cached
    ):
        use_cache = True

if use_cache:
    liquid_results = cached['liquid_results']
    diffusion_results = cached['diffusion_results']
    core_table = cached['core_table']
    compute_table = cached['compute_table']
    efficiency_table = cached['efficiency_table']
    print(f"Loaded cached evaluation from {EVAL_CACHE_PATH}")
else:
    liquid_results = evaluate_liquid(liquid_model, test_loader, device, NUM_EVAL_SAMPLES)
    diffusion_results = evaluate_diffusion(diffusion_model, test_loader, device, NUM_EVAL_SAMPLES)

    liquid_k10 = liquid_results['k_metrics']['10']
    diffusion_k10 = diffusion_results['k_metrics']['10']
    liquid_speedup = diffusion_results['k_sample_ms'] / max(liquid_results['k_sample_ms'], 1e-9)

    core_table = [
        {
            'Model': 'Liquid deterministic (0.5x params)',
            '# Params': liquid_params,
            'NLL ↓': None,
            'MSE ↓': liquid_results['deterministic']['mse'],
            'Best-of-10 MSE ↓': liquid_results['deterministic']['best_of_k_mse'],
            'Diversity ↑': 0.0,
            'Smoothness ↓': liquid_results['deterministic']['smoothness'],
            'Time (ms) ↓': liquid_results['single_pass_ms'],
        },
        {
            'Model': 'Liquid + MDN (K=10, 0.5x params)',
            '# Params': liquid_params,
            'NLL ↓': liquid_results['exact_nll'],
            'MSE ↓': liquid_k10['sample_mean_mse'],
            'Best-of-10 MSE ↓': liquid_k10['best_of_k_mse'],
            'Diversity ↑': liquid_k10['diversity_l2'],
            'Smoothness ↓': liquid_k10['smoothness'],
            'Time (ms) ↓': liquid_results['k_sample_ms'],
        },
        {
            'Model': 'Diffusion (K=10, 1.0x params)',
            '# Params': diffusion_params,
            'NLL ↓': diffusion_results['proxy_nll'],
            'MSE ↓': diffusion_k10['sample_mean_mse'],
            'Best-of-10 MSE ↓': diffusion_k10['best_of_k_mse'],
            'Diversity ↑': diffusion_k10['diversity_l2'],
            'Smoothness ↓': diffusion_k10['smoothness'],
            'Time (ms) ↓': diffusion_results['k_sample_ms'],
        },
    ]

    liquid_k10_time = liquid_results['k_sample_ms']
    compute_table = [
        {
            'Model': 'Liquid + MDN (K=10, 0.5x params)',
            'Relative Compute x': 1.0,
            'Speedup vs Diffusion ↑': liquid_speedup,
            'Best-of-10 MSE ↓': liquid_k10['best_of_k_mse'],
            'Time (ms) ↓': liquid_k10_time,
        },
        {
            'Model': 'Diffusion (K=10, 1.0x params)',
            'Relative Compute x': diffusion_results['k_sample_ms'] / max(liquid_k10_time, 1e-9),
            'Speedup vs Diffusion ↑': 1.0,
            'Best-of-10 MSE ↓': diffusion_k10['best_of_k_mse'],
            'Time (ms) ↓': diffusion_results['k_sample_ms'],
        },
    ]

    efficiency_table = [
        {
            'Model': 'Liquid + MDN',
            '# Params': liquid_params,
            'Params vs Diffusion x': liquid_params / max(diffusion_params, 1),
            'Latency Speedup vs Diffusion ↑': liquid_speedup,
            'NLL ↓': liquid_results['exact_nll'],
            'Best-of-10 MSE ↓': liquid_k10['best_of_k_mse'],
        },
        {
            'Model': 'Diffusion',
            '# Params': diffusion_params,
            'Params vs Diffusion x': 1.0,
            'Latency Speedup vs Diffusion ↑': 1.0,
            'NLL ↓': diffusion_results['proxy_nll'],
            'Best-of-10 MSE ↓': diffusion_k10['best_of_k_mse'],
        },
    ]

    epochs_completed = -1
    if os.path.exists(TRAINING_CACHE_PATH):
        try:
            epochs_completed = int(load_json(TRAINING_CACHE_PATH).get('epochs_completed', -1))
        except Exception:
            pass

    save_json(EVAL_CACHE_PATH, {
        'dataset': DATASET_NAME,
        'experiment_tag': EXPERIMENT_TAG,
        'num_diffusion_steps': int(NUM_DIFFUSION_STEPS),
        'num_eval_samples': int(NUM_EVAL_SAMPLES),
        'k_values': K_VALUES,
        'epochs_completed': int(epochs_completed),
        'liquid_checkpoint': LIQUID_CKPT_PATH,
        'diffusion_checkpoint': DIFFUSION_CKPT_PATH,
        'liquid_results': liquid_results,
        'diffusion_results': diffusion_results,
        'core_table': core_table,
        'compute_table': compute_table,
        'efficiency_table': efficiency_table,
    })
    print(f"Saved evaluation cache to {EVAL_CACHE_PATH}")

core_df = pd.DataFrame(core_table)
compute_df = pd.DataFrame(compute_table)
efficiency_df = pd.DataFrame(efficiency_table)

core_df.to_csv(f'{ARTIFACT_DIR}/offline_trajectory_modeling_performance_{ARTIFACT_STEM}.csv', index=False)
compute_df.to_csv(f'{ARTIFACT_DIR}/compute_normalized_performance_{ARTIFACT_STEM}.csv', index=False)
efficiency_df.to_csv(f'{ARTIFACT_DIR}/parameter_efficiency_{ARTIFACT_STEM}.csv', index=False)

print("\n" + "=" * 90)
print("Offline Trajectory Modeling Performance")
print("=" * 90)
print(core_df.to_string(index=False, na_rep='—', float_format=lambda x: f'{x:.6f}'))

print("\n" + "=" * 90)
print("Compute-Normalized Performance (latency proxy)")
print("=" * 90)
print(compute_df.to_string(index=False, float_format=lambda x: f'{x:.6f}'))

print("\n" + "=" * 90)
print("Parameter Efficiency Table")
print("=" * 90)
print(efficiency_df.to_string(index=False, float_format=lambda x: f'{x:.6f}'))

print("\nSaved tables:")
print(f"  - {ARTIFACT_DIR}/offline_trajectory_modeling_performance_{ARTIFACT_STEM}.csv")
print(f"  - {ARTIFACT_DIR}/compute_normalized_performance_{ARTIFACT_STEM}.csv")
print(f"  - {ARTIFACT_DIR}/parameter_efficiency_{ARTIFACT_STEM}.csv")

In [ ]:
# Ensure evaluation data is available
if 'liquid_results' not in globals() or 'diffusion_results' not in globals() or 'core_table' not in globals():
    if os.path.exists(EVAL_CACHE_PATH):
        cached_eval = load_json(EVAL_CACHE_PATH)
        liquid_results = cached_eval['liquid_results']
        diffusion_results = cached_eval['diffusion_results']
        core_table = cached_eval['core_table']
        compute_table = cached_eval['compute_table']
        efficiency_table = cached_eval['efficiency_table']
        print(f"Loaded eval cache from {EVAL_CACHE_PATH}")
    else:
        raise RuntimeError("No evaluation data in memory or cache. Run the evaluation cell first.")

ARTIFACT_STEM = f"{DATASET_NAME}_{EXPERIMENT_TAG}"
core_df = pd.DataFrame(core_table)
compute_df = pd.DataFrame(compute_table)
efficiency_df = pd.DataFrame(efficiency_table)
k_values = [int(k) for k in liquid_results['k_metrics'].keys()]
k_values.sort()
EPS = 1e-6

# ---- Table figure ----
fig_table, axes = plt.subplots(1, 2, figsize=(20, 5.5))
for ax in axes:
    ax.axis('off')

core_display = core_df.copy()
for col in core_display.columns[1:]:
    core_display[col] = core_display[col].map(lambda x: '—' if pd.isna(x) else f'{x:.4f}')

table1 = axes[0].table(
    cellText=core_display.values,
    colLabels=core_display.columns,
    loc='center',
    cellLoc='center'
)
table1.auto_set_font_size(False)
table1.set_fontsize(8.5)
table1.scale(1.12, 1.55)
axes[0].set_title('Offline Trajectory Modeling Performance', fontsize=12, fontweight='bold')

compute_display = compute_df.copy()
for col in compute_display.columns[1:]:
    compute_display[col] = compute_display[col].map(lambda x: f'{x:.4f}')

table2 = axes[1].table(
    cellText=compute_display.values,
    colLabels=compute_display.columns,
    loc='center',
    cellLoc='center'
)
table2.auto_set_font_size(False)
table2.set_fontsize(8.5)
table2.scale(1.12, 1.55)
axes[1].set_title('Compute-Normalized Performance (latency proxy)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{ARTIFACT_DIR}/offline_tables_{ARTIFACT_STEM}.png', dpi=150, bbox_inches='tight')
plt.show()

# ---- Plot 1: Best-of-K vs K ----
fig1, ax = plt.subplots(1, 1, figsize=(8, 5))
liquid_best = [liquid_results['k_metrics'][str(k)]['best_of_k_mse'] for k in k_values]
diff_best = [diffusion_results['k_metrics'][str(k)]['best_of_k_mse'] for k in k_values]
ax.plot(k_values, liquid_best, marker='o', linewidth=2, color='#1f77b4', label='Liquid + MDN (0.5x params)')
ax.plot(k_values, diff_best, marker='o', linewidth=2, color='#ff7f0e', label='Diffusion (1.0x params)')
ax.set_yscale('log')
ax.set_xlabel('Number of samples K')
ax.set_ylabel('Best-of-K MSE (log scale)')
ax.set_title('Best-of-K vs K')
ax.grid(True, alpha=0.3, which='both')
ax.legend()
for x, y in zip(k_values, liquid_best):
    ax.annotate(f'{y:.2e}', (x, y), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8, color='#1f77b4')
for x, y in zip(k_values, diff_best):
    ax.annotate(f'{y:.2e}', (x, y), textcoords='offset points', xytext=(0, -14), ha='center', fontsize=8, color='#ff7f0e')
plt.tight_layout()
plt.savefig(f'{ARTIFACT_DIR}/best_of_k_curve_{ARTIFACT_STEM}.png', dpi=150, bbox_inches='tight')
plt.show()

# ---- Plot 2: Qualitative trajectory samples ----
qual_n = min(NUM_QUAL_EXAMPLES, len(liquid_results['qualitative']['target']))
fig2, axes = plt.subplots(qual_n, 2, figsize=(12, 4 * qual_n))
if qual_n == 1:
    axes = np.array([axes])

for i in range(qual_n):
    gt = np.asarray(liquid_results['qualitative']['target'][i])
    liquid_det = np.asarray(liquid_results['qualitative']['deterministic'][i])
    liquid_samples = np.asarray(liquid_results['qualitative']['samples'][i])
    diffusion_proxy = np.asarray(diffusion_results['qualitative']['proxy_mean'][i])
    diffusion_samples = np.clip(np.asarray(diffusion_results['qualitative']['samples'][i]), -1.0, 1.0)

    combined = np.concatenate([
        gt,
        liquid_det,
        liquid_samples.reshape(-1, gt.shape[-1]),
        diffusion_proxy,
        diffusion_samples.reshape(-1, gt.shape[-1]),
    ], axis=0)
    xmin, ymin = combined.min(axis=0)
    xmax, ymax = combined.max(axis=0)
    xpad = max(0.05, 0.1 * (xmax - xmin))
    ypad = max(0.05, 0.1 * (ymax - ymin))

    ax = axes[i, 0]
    for sample in liquid_samples:
        ax.plot(sample[:, 0], sample[:, 1], color='#1f77b4', alpha=0.18)
    ax.plot(liquid_det[:, 0], liquid_det[:, 1], color='#1f77b4', linestyle='--', linewidth=2, label='Liquid mean')
    ax.plot(gt[:, 0], gt[:, 1], color='black', linewidth=2.5, label='Ground truth')
    ax.set_xlim(xmin - xpad, xmax + xpad)
    ax.set_ylim(ymin - ypad, ymax + ypad)
    ax.set_title(f'Example {i+1}: Liquid samples')
    ax.set_xlabel('action[0]')
    ax.set_ylabel('action[1]')
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend()

    ax = axes[i, 1]
    for sample in diffusion_samples:
        ax.plot(sample[:, 0], sample[:, 1], color='#ff7f0e', alpha=0.18)
    ax.plot(diffusion_proxy[:, 0], diffusion_proxy[:, 1], color='#ff7f0e', linestyle='--', linewidth=2, label='Diffusion proxy mean')
    ax.plot(gt[:, 0], gt[:, 1], color='black', linewidth=2.5, label='Ground truth')
    ax.set_xlim(xmin - xpad, xmax + xpad)
    ax.set_ylim(ymin - ypad, ymax + ypad)
    ax.set_title(f'Example {i+1}: Diffusion samples')
    ax.set_xlabel('action[0]')
    ax.set_ylabel('action[1]')
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend()

plt.tight_layout()
plt.savefig(f'{ARTIFACT_DIR}/qualitative_trajectory_samples_{ARTIFACT_STEM}.png', dpi=150, bbox_inches='tight')
plt.show()

# ---- Plot 3: Error vs horizon ----
fig3, ax = plt.subplots(1, 1, figsize=(9, 5))
horizon = np.arange(1, len(liquid_results['deterministic']['per_horizon_mse']) + 1)
ax.plot(horizon, liquid_results['deterministic']['per_horizon_mse'], color='#1f77b4', linestyle='--', linewidth=2, label='Liquid deterministic')
ax.plot(horizon, liquid_results['k_metrics']['10']['per_horizon_best_of_k_mse'], color='#1f77b4', linewidth=2, label='Liquid best-of-10')
ax.plot(horizon, diffusion_results['k_metrics']['10']['per_horizon_best_of_k_mse'], color='#ff7f0e', linewidth=2, label='Diffusion best-of-10')
ax.set_yscale('log')
ax.set_xlabel('Prediction horizon step')
ax.set_ylabel('MSE (log scale)')
ax.set_title('Error vs Horizon')
ax.grid(True, alpha=0.3, which='both')
ax.legend()
plt.tight_layout()
plt.savefig(f'{ARTIFACT_DIR}/error_vs_horizon_{ARTIFACT_STEM}.png', dpi=150, bbox_inches='tight')
plt.show()

# ---- Plot 4: Diversity vs error tradeoff ----
fig4, ax = plt.subplots(1, 1, figsize=(8, 5))
liquid_div = np.asarray(liquid_results['k_metrics']['10']['per_example_diversity']) + EPS
liquid_err = np.asarray(liquid_results['k_metrics']['10']['per_example_best_of_k_mse']) + EPS
diff_div = np.asarray(diffusion_results['k_metrics']['10']['per_example_diversity']) + EPS
diff_err = np.asarray(diffusion_results['k_metrics']['10']['per_example_best_of_k_mse']) + EPS
ax.scatter(liquid_div, liquid_err, color='#1f77b4', alpha=0.35, s=18, label='Liquid + MDN (0.5x params)')
ax.scatter(diff_div, diff_err, color='#ff7f0e', alpha=0.35, s=18, label='Diffusion (1.0x params)')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Diversity (log scale)')
ax.set_ylabel('Best-of-10 MSE (log scale)')
ax.set_title('Diversity vs Error Tradeoff')
ax.grid(True, alpha=0.3, which='both')
ax.legend()
plt.tight_layout()
plt.savefig(f'{ARTIFACT_DIR}/diversity_vs_error_{ARTIFACT_STEM}.png', dpi=150, bbox_inches='tight')
plt.show()

# ---- Training curves ----
history_data = None
if os.path.exists(TRAINING_CACHE_PATH):
    history_data = load_json(TRAINING_CACHE_PATH)
    print(f"Loaded training cache from {TRAINING_CACHE_PATH}")

if history_data is not None and len(history_data.get('epochs', [])) > 0:
    epochs = history_data['epochs']
    fig5, ax = plt.subplots(1, 1, figsize=(10, 4.5))

    train_liquid = bool(history_data.get('train_liquid', True))
    if train_liquid:
        ax.plot(epochs, history_data['liquid_train_nll'], color='#1f77b4', label='Liquid Train NLL')
        ax.plot(epochs, history_data['liquid_val_nll'], color='#1f77b4', linestyle='--', label='Liquid Val NLL')
    else:
        liquid_train = history_data['liquid_train_nll'][-1]
        liquid_val = history_data['liquid_val_nll'][-1]
        ax.axhline(liquid_train, color='#1f77b4', linewidth=2, label='Liquid frozen train baseline')
        ax.axhline(liquid_val, color='#1f77b4', linestyle='--', linewidth=2, label='Liquid frozen val baseline')

    ax.plot(epochs, history_data['diffusion_train_nll'], color='#ff7f0e', label='Diffusion Train NLL')
    ax.plot(epochs, history_data['diffusion_val_nll'], color='#ff7f0e', linestyle='--', label='Diffusion Val NLL')
    if 'diffusion_train_ddpm' in history_data:
        ax.plot(epochs, history_data['diffusion_train_ddpm'], color='#2ca02c', alpha=0.8, label='Diffusion Train DDPM')
        ax.plot(epochs, history_data['diffusion_val_ddpm'], color='#2ca02c', linestyle='--', alpha=0.8, label='Diffusion Val DDPM')

    title = 'Fair Training Progress (Liquid at Half Parameter Count)' if train_liquid else 'Diffusion Repair Progress (Liquid Frozen Reference)'
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss / NLL')
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=2)
    plt.tight_layout()
    plt.savefig(f'{ARTIFACT_DIR}/training_progress_{ARTIFACT_STEM}.png', dpi=150, bbox_inches='tight')
    plt.show()

print('✓ Requested tables and plots saved')
print('\nSaved plot artifacts:')
print(f'  - {ARTIFACT_DIR}/offline_tables_{ARTIFACT_STEM}.png')
print(f'  - {ARTIFACT_DIR}/best_of_k_curve_{ARTIFACT_STEM}.png')
print(f'  - {ARTIFACT_DIR}/qualitative_trajectory_samples_{ARTIFACT_STEM}.png')
print(f'  - {ARTIFACT_DIR}/error_vs_horizon_{ARTIFACT_STEM}.png')
print(f'  - {ARTIFACT_DIR}/diversity_vs_error_{ARTIFACT_STEM}.png')
print(f'  - {ARTIFACT_DIR}/training_progress_{ARTIFACT_STEM}.png')

In [ ]:
# ---- Multi-dataset extension: RoboMimic Can + cross-dataset comparisons ----

import h5py

from pathlib import Path

from matplotlib.lines import Line2D



MULTI_DATASET_TAG = EXPERIMENT_TAG

PUSHT_CACHE_PATH = Path(ARTIFACT_DIR) / f"eval_results_pusht_{MULTI_DATASET_TAG}.json"

PUSHT_HISTORY_PATH = Path(ARTIFACT_DIR) / f"training_history_pusht_{MULTI_DATASET_TAG}.json"

ROBOMIMIC_CAN_DATASET_PATH = Path("datasets/robomimic/v1.5/can/ph/low_dim_v15.hdf5")

ROBOMIMIC_CAN_NAME = "robomimic_can"

ROBOMIMIC_CAN_LABEL = "RoboMimic Can"

PUSHT_LABEL = "push_t"

ROBOMIMIC_IMAGE_SHAPE = (96, 96, 3)

ROBOMIMIC_OBS_KEYS = [

    'object',

    'robot0_eef_pos',

    'robot0_eef_quat',

    'robot0_eef_quat_site',

    'robot0_gripper_qpos',

    'robot0_gripper_qvel',

    'robot0_joint_pos',

    'robot0_joint_pos_cos',

    'robot0_joint_pos_sin',

    'robot0_joint_vel',

]

ROBOMIMIC_NUM_EPOCHS = 120

ROBOMIMIC_PATIENCE = 999

ROBOMIMIC_BATCH_SIZE = 32





def _decode_demo_keys(arr):

    keys = []

    for item in arr:

        if isinstance(item, bytes):

            keys.append(item.decode('utf-8'))

        else:

            keys.append(str(item))

    return keys





def _concat_robomimic_obs(obs_group):

    parts = []

    for key in ROBOMIMIC_OBS_KEYS:

        parts.append(obs_group[key][:].astype(np.float32))

    return np.concatenate(parts, axis=-1)





class RoboMimicCanDatasetWithCLIP(Dataset):

    def __init__(self, action_array, state_array, indices, pred_horizon=16, image_shape=(96, 96, 3)):

        self.action_array = action_array.astype(np.float32)

        self.state_array = state_array.astype(np.float32)

        self.indices = indices.astype(np.int64)

        self.pred_horizon = pred_horizon

        self.image_shape = image_shape

        self.zero_image = np.zeros(image_shape, dtype=np.float32)



    def __len__(self):

        return len(self.indices)



    def __getitem__(self, idx):

        buffer_start_idx, buffer_end_idx, sample_start_idx, sample_end_idx = self.indices[idx]

        nsample = sample_sequence(

            train_data={

                'action': self.action_array,

                'state': self.state_array,

            },

            sequence_length=self.pred_horizon,

            buffer_start_idx=int(buffer_start_idx),

            buffer_end_idx=int(buffer_end_idx),

            sample_start_idx=int(sample_start_idx),

            sample_end_idx=int(sample_end_idx),

        )

        image = np.repeat(self.zero_image[None, ...], self.pred_horizon, axis=0)

        return {

            'image': torch.from_numpy(image).float(),

            'state': torch.from_numpy(nsample['state']).float(),

            'action': torch.from_numpy(nsample['action']).float(),

        }





def build_robomimic_can_loaders(

    dataset_path: Path,

    pred_horizon: int = 16,

    obs_horizon: int = 2,

    action_horizon: int = 8,

    batch_size: int = 32,

    seed: int = 7,

):

    if not dataset_path.exists():

        raise FileNotFoundError(f"RoboMimic Can dataset not found at {dataset_path}")



    with h5py.File(dataset_path, 'r') as f:

        train_demo_keys = _decode_demo_keys(f['mask']['train'][:])

        test_demo_keys = _decode_demo_keys(f['mask']['valid'][:])



        rng = np.random.default_rng(seed)

        perm = rng.permutation(len(train_demo_keys))

        train_count = max(1, int(0.85 * len(train_demo_keys)))

        inner_train_keys = [train_demo_keys[i] for i in perm[:train_count]]

        val_keys = [train_demo_keys[i] for i in perm[train_count:]]

        if len(val_keys) == 0:

            val_keys = inner_train_keys[-max(1, len(inner_train_keys) // 10):]

            inner_train_keys = inner_train_keys[:-len(val_keys)]



        def load_split(keys):

            action_parts = []

            state_parts = []

            lengths = []

            for key in keys:

                demo = f['data'][key]

                actions = demo['actions'][:].astype(np.float32)

                states = _concat_robomimic_obs(demo['obs'])

                action_parts.append(actions)

                state_parts.append(states)

                lengths.append(actions.shape[0])

            return action_parts, state_parts, lengths



        train_action_parts, train_state_parts, train_lengths = load_split(inner_train_keys)

        val_action_parts, val_state_parts, val_lengths = load_split(val_keys)

        test_action_parts, test_state_parts, test_lengths = load_split(test_demo_keys)



    train_action_full = np.concatenate(train_action_parts, axis=0)

    train_state_full = np.concatenate(train_state_parts, axis=0)

    stats = {

        'action': get_data_stats(train_action_full),

        'state': get_data_stats(train_state_full),

    }



    def pack_dataset(action_parts, state_parts, lengths):

        actions = np.concatenate([normalize_data(arr, stats['action']) for arr in action_parts], axis=0)

        states = np.concatenate([normalize_data(arr, stats['state']) for arr in state_parts], axis=0)

        episode_ends = np.cumsum(lengths)

        indices = create_sample_indices(

            episode_ends=episode_ends,

            sequence_length=pred_horizon,

            pad_before=obs_horizon - 1,

            pad_after=action_horizon - 1,

        )

        dataset = RoboMimicCanDatasetWithCLIP(

            action_array=actions,

            state_array=states,

            indices=indices,

            pred_horizon=pred_horizon,

            image_shape=ROBOMIMIC_IMAGE_SHAPE,

        )

        return dataset, actions.shape[-1], states.shape[-1], len(lengths)



    train_dataset, action_dim, state_dim, train_demos = pack_dataset(train_action_parts, train_state_parts, train_lengths)

    val_dataset, _, _, val_demos = pack_dataset(val_action_parts, val_state_parts, val_lengths)

    test_dataset, _, _, test_demos = pack_dataset(test_action_parts, test_state_parts, test_lengths)



    generator = torch.Generator().manual_seed(seed)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True, generator=generator)

    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)



    metadata = {

        'dataset_name': ROBOMIMIC_CAN_NAME,

        'display_name': ROBOMIMIC_CAN_LABEL,

        'state_dim': state_dim,

        'action_dim': action_dim,

        'train_demos': train_demos,

        'val_demos': val_demos,

        'test_demos': test_demos,

        'train_windows': len(train_dataset),

        'val_windows': len(val_dataset),

        'test_windows': len(test_dataset),

        'pred_horizon': pred_horizon,

        'obs_horizon': obs_horizon,

        'action_horizon': action_horizon,

        'dataset_path': str(dataset_path),

    }

    return train_loader, val_loader, test_loader, metadata





def run_dataset_benchmark(

    dataset_name: str,

    display_name: str,

    train_loader,

    val_loader,

    test_loader,

    state_dim: int,

    action_dim: int,

    num_epochs: int = 120,

    patience: int = 999,

    batch_size: int = 32,

):

    checkpoint_liquid = Path(CHECKPOINT_DIR) / f"liquid_jepa_{dataset_name}_{MULTI_DATASET_TAG}_best.pt"

    checkpoint_diffusion = Path(CHECKPOINT_DIR) / f"diffusion_jepa_{dataset_name}_{MULTI_DATASET_TAG}_best.pt"

    training_cache = Path(ARTIFACT_DIR) / f"training_history_{dataset_name}_{MULTI_DATASET_TAG}.json"

    eval_cache = Path(ARTIFACT_DIR) / f"eval_results_{dataset_name}_{MULTI_DATASET_TAG}.json"



    dataset_backbone = SharedBackbone(

        clip_model=clip_model,

        clip_feat_dim=CLIP_FEAT_DIM,

        state_dim=state_dim,

        hidden_dim=D_MODEL,

        num_layers=4,

    ).to(device)



    liquid_model_local = LiquidTrajectoryModel(

        backbone=copy.deepcopy(dataset_backbone).to(device),

        action_dim=action_dim,

        hidden_dim=LIQUID_HIDDEN_SIZE,

        seq_length=16,

        num_layers=NUM_LAYERS,

        num_mixtures=NUM_MIXTURES,

        dropout=DROPOUT,

    ).to(device)

    diffusion_model_local = DiffusionTrajectoryModel(

        backbone=copy.deepcopy(dataset_backbone).to(device),

        action_dim=action_dim,

        hidden_dim=DIFFUSION_HIDDEN_SIZE,

        seq_length=16,

        num_diffusion_steps=NUM_DIFFUSION_STEPS,

        num_mixtures=NUM_MIXTURES,

    ).to(device)



    liquid_optimizer_local = torch.optim.AdamW(liquid_model_local.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    diffusion_optimizer_local = torch.optim.AdamW(diffusion_model_local.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    liquid_scheduler_local = create_scheduler(liquid_optimizer_local, num_epochs, WARMUP_EPOCHS)

    diffusion_scheduler_local = create_scheduler(diffusion_optimizer_local, num_epochs, WARMUP_EPOCHS)



    best_liquid = float('inf')

    best_diffusion = float('inf')

    liquid_no_improve_local = 0

    diffusion_no_improve_local = 0



    history = {

        'dataset': dataset_name,

        'display_name': display_name,

        'batch_size': batch_size,

        'num_epochs_config': int(num_epochs),

        'num_diffusion_steps': int(NUM_DIFFUSION_STEPS),

        'train_liquid': True,

        'train_diffusion': True,

        'diffusion_nll_weight': float(DIFFUSION_NLL_WEIGHT),

        'diffusion_ddpm_weight': float(DIFFUSION_DDPM_WEIGHT),

        'liquid_checkpoint': str(checkpoint_liquid),

        'diffusion_checkpoint': str(checkpoint_diffusion),

        'epochs': [],

        'teacher_forcing_ratio': [],

        'free_running_weight': [],

        'liquid_train_nll': [],

        'liquid_val_nll': [],

        'diffusion_train_nll': [],

        'diffusion_val_nll': [],

        'diffusion_train_ddpm': [],

        'diffusion_val_ddpm': [],

        'diffusion_train_total': [],

        'diffusion_val_total': [],

        'liquid_lr': [],

        'diffusion_lr': [],

        'epoch_time_sec': [],

    }



    print(f"\n{'=' * 88}")

    print(f"Training shared JEPA benchmark on {display_name}")

    print(f"  state dim: {state_dim} | action dim: {action_dim} | train windows: {len(train_loader.dataset)}")

    print(f"{'=' * 88}")



    for epoch in range(num_epochs):

        epoch_start = time.perf_counter()

        progress = epoch / max(1, num_epochs - 1)

        teacher_forcing_ratio = max(0.15, 1.0 - 0.90 * progress)

        free_w = min(0.75, 0.20 + 0.65 * progress)

        tf_w = 1.0 - free_w



        liquid_model_local.train()

        diffusion_model_local.train()

        liquid_train_total = 0.0

        diffusion_train_nll = 0.0

        diffusion_train_ddpm = 0.0

        diffusion_train_total = 0.0



        with tqdm(train_loader, desc=f"[{display_name}] Train {epoch}", leave=False) as pbar:

            for batch in pbar:

                images = batch['image'].to(device)

                state = batch['state'].to(device)

                action_target = batch['action'].to(device)



                liquid_optimizer_local.zero_grad()

                out_tf = liquid_model_local(images, state, action_target=action_target, teacher_forcing_ratio=teacher_forcing_ratio, return_mdn=True)

                loss_tf = mdn_nll_loss(out_tf['logits'], out_tf['mu'], out_tf['log_sigma'], action_target)

                out_free = liquid_model_local(images, state, action_target=None, teacher_forcing_ratio=0.0, return_mdn=True)

                loss_free = mdn_nll_loss(out_free['logits'], out_free['mu'], out_free['log_sigma'], action_target)

                liquid_loss = tf_w * loss_tf + free_w * loss_free

                liquid_loss.backward()

                torch.nn.utils.clip_grad_norm_(liquid_model_local.parameters(), 1.0)

                liquid_optimizer_local.step()

                liquid_train_total += liquid_loss.item()



                diffusion_optimizer_local.zero_grad()

                out = diffusion_model_local(images, state, action_target=action_target, return_mdn=True, inference=False)

                loss_nll = mdn_nll_loss(out['logits'], out['mu'], out['log_sigma'], action_target)

                loss_ddpm = F.mse_loss(out['pred_epsilon'], out['target_epsilon'])

                diffusion_loss = DIFFUSION_NLL_WEIGHT * loss_nll + DIFFUSION_DDPM_WEIGHT * loss_ddpm

                diffusion_loss.backward()

                torch.nn.utils.clip_grad_norm_(diffusion_model_local.parameters(), 1.0)

                diffusion_optimizer_local.step()

                diffusion_train_nll += loss_nll.item()

                diffusion_train_ddpm += loss_ddpm.item()

                diffusion_train_total += diffusion_loss.item()



                pbar.set_postfix(liquid=f"{liquid_loss.item():.4f}", diff=f"{diffusion_loss.item():.4f}")



        avg_liquid_train = liquid_train_total / len(train_loader)

        avg_diff_train_nll = diffusion_train_nll / len(train_loader)

        avg_diff_train_ddpm = diffusion_train_ddpm / len(train_loader)

        avg_diff_train_total = diffusion_train_total / len(train_loader)



        liquid_model_local.eval()

        diffusion_model_local.eval()

        liquid_val_nll = 0.0

        diffusion_val_nll = 0.0

        diffusion_val_ddpm = 0.0

        diffusion_val_total = 0.0



        with torch.no_grad():

            for batch in val_loader:

                images = batch['image'].to(device)

                state = batch['state'].to(device)

                action_target = batch['action'].to(device)



                out_liquid = liquid_model_local(images, state, teacher_forcing_ratio=0.0, return_mdn=True)

                liquid_val_nll += mdn_nll_loss(out_liquid['logits'], out_liquid['mu'], out_liquid['log_sigma'], action_target).item()



                out_diffusion = diffusion_model_local(images, state, action_target=action_target, return_mdn=True, inference=False)

                diff_val_nll = mdn_nll_loss(out_diffusion['logits'], out_diffusion['mu'], out_diffusion['log_sigma'], action_target)

                diff_val_ddpm = F.mse_loss(out_diffusion['pred_epsilon'], out_diffusion['target_epsilon'])

                diffusion_val_nll += diff_val_nll.item()

                diffusion_val_ddpm += diff_val_ddpm.item()

                diffusion_val_total += (DIFFUSION_NLL_WEIGHT * diff_val_nll + DIFFUSION_DDPM_WEIGHT * diff_val_ddpm).item()



        avg_liquid_val = liquid_val_nll / len(val_loader)

        avg_diff_val_nll = diffusion_val_nll / len(val_loader)

        avg_diff_val_ddpm = diffusion_val_ddpm / len(val_loader)

        avg_diff_val_total = diffusion_val_total / len(val_loader)



        liquid_scheduler_local.step()

        diffusion_scheduler_local.step()



        liquid_tag_local = ''

        diffusion_tag_local = ''

        if avg_liquid_val < (best_liquid - MIN_DELTA):

            best_liquid = avg_liquid_val

            liquid_no_improve_local = 0

            torch.save(liquid_model_local.state_dict(), checkpoint_liquid)

            liquid_tag_local = ' [best]'

        else:

            liquid_no_improve_local += 1



        if avg_diff_val_total < (best_diffusion - MIN_DELTA):

            best_diffusion = avg_diff_val_total

            diffusion_no_improve_local = 0

            torch.save(diffusion_model_local.state_dict(), checkpoint_diffusion)

            diffusion_tag_local = ' [best]'

        else:

            diffusion_no_improve_local += 1



        epoch_time = time.perf_counter() - epoch_start

        history['epochs'].append(int(epoch))

        history['teacher_forcing_ratio'].append(float(teacher_forcing_ratio))

        history['free_running_weight'].append(float(free_w))

        history['liquid_train_nll'].append(float(avg_liquid_train))

        history['liquid_val_nll'].append(float(avg_liquid_val))

        history['diffusion_train_nll'].append(float(avg_diff_train_nll))

        history['diffusion_val_nll'].append(float(avg_diff_val_nll))

        history['diffusion_train_ddpm'].append(float(avg_diff_train_ddpm))

        history['diffusion_val_ddpm'].append(float(avg_diff_val_ddpm))

        history['diffusion_train_total'].append(float(avg_diff_train_total))

        history['diffusion_val_total'].append(float(avg_diff_val_total))

        history['liquid_lr'].append(float(liquid_optimizer_local.param_groups[0]['lr']))

        history['diffusion_lr'].append(float(diffusion_optimizer_local.param_groups[0]['lr']))

        history['epoch_time_sec'].append(float(epoch_time))

        save_json(str(training_cache), history)



        print(f"Epoch {epoch} | {display_name} | Time: {epoch_time:.1f}s")

        print(f"  [Liquid]    Train NLL: {avg_liquid_train:.5f} | Val NLL: {avg_liquid_val:.5f}{liquid_tag_local}")

        print(f"  [Diffusion] Train total: {avg_diff_train_total:.5f} | Val total: {avg_diff_val_total:.5f}{diffusion_tag_local}")

        print(f"               Train NLL: {avg_diff_train_nll:.5f} | Val NLL: {avg_diff_val_nll:.5f}")

        print(f"               Train DDPM: {avg_diff_train_ddpm:.5f} | Val DDPM: {avg_diff_val_ddpm:.5f}")



    history['best_liquid_val_nll'] = float(best_liquid)

    history['best_diffusion_val_total'] = float(best_diffusion)

    history['epochs_completed'] = len(history['epochs'])

    save_json(str(training_cache), history)



    liquid_model_local.load_state_dict(torch.load(checkpoint_liquid, map_location=device))

    diffusion_model_local.load_state_dict(torch.load(checkpoint_diffusion, map_location=device))

    liquid_model_local.eval()

    diffusion_model_local.eval()



    liquid_results_local = evaluate_liquid(liquid_model_local, test_loader, device, NUM_EVAL_SAMPLES)

    diffusion_results_local = evaluate_diffusion(diffusion_model_local, test_loader, device, NUM_EVAL_SAMPLES)

    liquid_k10 = liquid_results_local['k_metrics']['10']

    diffusion_k10 = diffusion_results_local['k_metrics']['10']

    liquid_speedup_local = diffusion_results_local['k_sample_ms'] / max(liquid_results_local['k_sample_ms'], 1e-9)



    liquid_params_local = sum(p.numel() for p in liquid_model_local.parameters())

    diffusion_params_local = sum(p.numel() for p in diffusion_model_local.parameters())



    core_table_local = [

        {

            'Dataset': display_name,

            'Model': 'Liquid deterministic (0.5x params)',

            '# Params': liquid_params_local,

            'NLL ↓': None,

            'MSE ↓': liquid_results_local['deterministic']['mse'],

            'Best-of-10 MSE ↓': liquid_results_local['deterministic']['best_of_k_mse'],

            'Diversity ↑': 0.0,

            'Smoothness ↓': liquid_results_local['deterministic']['smoothness'],

            'Time (ms) ↓': liquid_results_local['single_pass_ms'],

        },

        {

            'Dataset': display_name,

            'Model': 'Liquid + MDN (K=10, 0.5x params)',

            '# Params': liquid_params_local,

            'NLL ↓': liquid_results_local['exact_nll'],

            'MSE ↓': liquid_k10['sample_mean_mse'],

            'Best-of-10 MSE ↓': liquid_k10['best_of_k_mse'],

            'Diversity ↑': liquid_k10['diversity_l2'],

            'Smoothness ↓': liquid_k10['smoothness'],

            'Time (ms) ↓': liquid_results_local['k_sample_ms'],

        },

        {

            'Dataset': display_name,

            'Model': 'Diffusion (K=10, 1.0x params)',

            '# Params': diffusion_params_local,

            'NLL ↓': diffusion_results_local['proxy_nll'],

            'MSE ↓': diffusion_k10['sample_mean_mse'],

            'Best-of-10 MSE ↓': diffusion_k10['best_of_k_mse'],

            'Diversity ↑': diffusion_k10['diversity_l2'],

            'Smoothness ↓': diffusion_k10['smoothness'],

            'Time (ms) ↓': diffusion_results_local['k_sample_ms'],

        },

    ]

    compute_table_local = [

        {

            'Dataset': display_name,

            'Model': 'Liquid + MDN (K=10, 0.5x params)',

            'Relative Compute x': 1.0,

            'Speedup vs Diffusion ↑': liquid_speedup_local,

            'Best-of-10 MSE ↓': liquid_k10['best_of_k_mse'],

            'Time (ms) ↓': liquid_results_local['k_sample_ms'],

        },

        {

            'Dataset': display_name,

            'Model': 'Diffusion (K=10, 1.0x params)',

            'Relative Compute x': diffusion_results_local['k_sample_ms'] / max(liquid_results_local['k_sample_ms'], 1e-9),

            'Speedup vs Diffusion ↑': 1.0,

            'Best-of-10 MSE ↓': diffusion_k10['best_of_k_mse'],

            'Time (ms) ↓': diffusion_results_local['k_sample_ms'],

        },

    ]

    efficiency_table_local = [

        {

            'Dataset': display_name,

            'Model': 'Liquid + MDN',

            '# Params': liquid_params_local,

            'Params vs Diffusion x': liquid_params_local / max(diffusion_params_local, 1),

            'Latency Speedup vs Diffusion ↑': liquid_speedup_local,

            'NLL ↓': liquid_results_local['exact_nll'],

            'Best-of-10 MSE ↓': liquid_k10['best_of_k_mse'],

        },

        {

            'Dataset': display_name,

            'Model': 'Diffusion',

            '# Params': diffusion_params_local,

            'Params vs Diffusion x': 1.0,

            'Latency Speedup vs Diffusion ↑': 1.0,

            'NLL ↓': diffusion_results_local['proxy_nll'],

            'Best-of-10 MSE ↓': diffusion_k10['best_of_k_mse'],

        },

    ]



    payload = {

        'dataset': dataset_name,

        'display_name': display_name,

        'experiment_tag': MULTI_DATASET_TAG,

        'num_diffusion_steps': int(NUM_DIFFUSION_STEPS),

        'num_eval_samples': int(NUM_EVAL_SAMPLES),

        'k_values': K_VALUES,

        'epochs_completed': int(history['epochs_completed']),

        'liquid_checkpoint': str(checkpoint_liquid),

        'diffusion_checkpoint': str(checkpoint_diffusion),

        'liquid_results': liquid_results_local,

        'diffusion_results': diffusion_results_local,

        'core_table': core_table_local,

        'compute_table': compute_table_local,

        'efficiency_table': efficiency_table_local,

        'state_dim': int(state_dim),

        'action_dim': int(action_dim),

    }

    save_json(str(eval_cache), payload)



    pd.DataFrame(core_table_local).to_csv(Path(ARTIFACT_DIR) / f"offline_trajectory_modeling_performance_{dataset_name}_{MULTI_DATASET_TAG}.csv", index=False)

    pd.DataFrame(compute_table_local).to_csv(Path(ARTIFACT_DIR) / f"compute_normalized_performance_{dataset_name}_{MULTI_DATASET_TAG}.csv", index=False)

    pd.DataFrame(efficiency_table_local).to_csv(Path(ARTIFACT_DIR) / f"parameter_efficiency_{dataset_name}_{MULTI_DATASET_TAG}.csv", index=False)



    return payload, history





print('✓ Multi-dataset helper functions ready')

print(f'  Push-T cache: {PUSHT_CACHE_PATH}')

print(f'  RoboMimic Can path: {ROBOMIMIC_CAN_DATASET_PATH}')


In [ ]:
# ---- PointMaze dataset support ----

import minari



POINTMAZE_DATASET_ID = "D4RL/pointmaze/umaze-v2"

POINTMAZE_NAME = "pointmaze_umaze"

POINTMAZE_LABEL = "PointMaze"

POINTMAZE_IMAGE_SHAPE = (96, 96, 3)

POINTMAZE_OBS_KEYS = ['observation', 'achieved_goal', 'desired_goal']

POINTMAZE_NUM_EPOCHS = 120

POINTMAZE_PATIENCE = 999

POINTMAZE_BATCH_SIZE = 128

POINTMAZE_MAX_TRAIN_EPISODES = 1024

POINTMAZE_MAX_VAL_EPISODES = 128

POINTMAZE_MAX_TEST_EPISODES = 128





def _concat_pointmaze_obs(obs_dict, traj_len=None):

    parts = []

    for key in POINTMAZE_OBS_KEYS:

        arr = np.asarray(obs_dict[key], dtype=np.float32)

        if traj_len is not None:

            arr = arr[:traj_len]

        parts.append(arr)

    return np.concatenate(parts, axis=-1)





class PointMazeDatasetWithCLIP(Dataset):

    def __init__(self, action_array, state_array, indices, pred_horizon=16, image_shape=(96, 96, 3)):

        self.action_array = action_array.astype(np.float32)

        self.state_array = state_array.astype(np.float32)

        self.indices = indices.astype(np.int64)

        self.pred_horizon = pred_horizon

        self.image_shape = image_shape

        self.zero_image = np.zeros(image_shape, dtype=np.float32)



    def __len__(self):

        return len(self.indices)



    def __getitem__(self, idx):

        buffer_start_idx, buffer_end_idx, sample_start_idx, sample_end_idx = self.indices[idx]

        nsample = sample_sequence(

            train_data={

                'action': self.action_array,

                'state': self.state_array,

            },

            sequence_length=self.pred_horizon,

            buffer_start_idx=int(buffer_start_idx),

            buffer_end_idx=int(buffer_end_idx),

            sample_start_idx=int(sample_start_idx),

            sample_end_idx=int(sample_end_idx),

        )

        image = np.repeat(self.zero_image[None, ...], self.pred_horizon, axis=0)

        return {

            'image': torch.from_numpy(image).float(),

            'state': torch.from_numpy(nsample['state']).float(),

            'action': torch.from_numpy(nsample['action']).float(),

        }





def build_pointmaze_loaders(

    dataset_id: str,

    pred_horizon: int = 16,

    obs_horizon: int = 2,

    action_horizon: int = 8,

    batch_size: int = 128,

    seed: int = 7,

    max_train_episodes: int = POINTMAZE_MAX_TRAIN_EPISODES,

    max_val_episodes: int = POINTMAZE_MAX_VAL_EPISODES,

    max_test_episodes: int = POINTMAZE_MAX_TEST_EPISODES,

):

    dataset = minari.load_dataset(dataset_id, download=True)

    total_episodes = int(dataset.total_episodes)

    if total_episodes < 3:

        raise ValueError(f"PointMaze dataset {dataset_id} has too few episodes: {total_episodes}")



    # --- Fixed split: always compute val/test from the FULL dataset first, ---

    # --- then subset the training pool.  This guarantees the same val/test  ---

    # --- episodes regardless of max_train_episodes (sample-eff fractions). ---

    rng = np.random.default_rng(seed)

    permutation = rng.permutation(total_episodes)

    full_train_count = max(1, int(round(0.80 * total_episodes)))

    val_count = max(1, min(max_val_episodes, int(round(0.10 * total_episodes))))

    remaining = total_episodes - full_train_count - val_count

    test_count = max(1, min(max_test_episodes, remaining if remaining > 0 else 1))

    if full_train_count + val_count + test_count > total_episodes:

        overflow = full_train_count + val_count + test_count - total_episodes

        test_count = max(1, test_count - overflow)

    # Val and test are ALWAYS the same slices of the permutation

    val_idx = permutation[full_train_count:full_train_count + val_count].tolist()

    test_idx = permutation[full_train_count + val_count:full_train_count + val_count + test_count].tolist()

    if len(test_idx) == 0:

        test_idx = permutation[-1:].tolist()

    # Training: take up to max_train_episodes from the fixed training pool

    full_train_pool = permutation[:full_train_count]

    train_count = max(1, min(max_train_episodes, full_train_count))

    train_idx = full_train_pool[:train_count].tolist()



    def load_split(indices):

        action_parts = []

        state_parts = []

        lengths = []

        for episode in dataset.iterate_episodes(episode_indices=indices):

            actions = np.asarray(episode.actions, dtype=np.float32)

            traj_len = actions.shape[0]

            if traj_len < 2:

                continue

            states = _concat_pointmaze_obs(episode.observations, traj_len=traj_len)

            action_parts.append(actions)

            state_parts.append(states)

            lengths.append(traj_len)

        if len(action_parts) == 0:

            raise ValueError(f"No usable PointMaze episodes found for indices: {indices[:5]}")

        return action_parts, state_parts, lengths



    train_action_parts, train_state_parts, train_lengths = load_split(train_idx)

    val_action_parts, val_state_parts, val_lengths = load_split(val_idx)

    test_action_parts, test_state_parts, test_lengths = load_split(test_idx)



    train_action_full = np.concatenate(train_action_parts, axis=0)

    train_state_full = np.concatenate(train_state_parts, axis=0)

    stats = {

        'action': get_data_stats(train_action_full),

        'state': get_data_stats(train_state_full),

    }



    def pack_dataset(action_parts, state_parts, lengths):

        actions = np.concatenate([normalize_data(arr, stats['action']) for arr in action_parts], axis=0)

        states = np.concatenate([normalize_data(arr, stats['state']) for arr in state_parts], axis=0)

        episode_ends = np.cumsum(lengths)

        indices = create_sample_indices(

            episode_ends=episode_ends,

            sequence_length=pred_horizon,

            pad_before=obs_horizon - 1,

            pad_after=action_horizon - 1,

        )

        dataset_local = PointMazeDatasetWithCLIP(

            action_array=actions,

            state_array=states,

            indices=indices,

            pred_horizon=pred_horizon,

            image_shape=POINTMAZE_IMAGE_SHAPE,

        )

        return dataset_local, actions.shape[-1], states.shape[-1], len(lengths)



    train_dataset, action_dim, state_dim, train_demos = pack_dataset(train_action_parts, train_state_parts, train_lengths)

    val_dataset, _, _, val_demos = pack_dataset(val_action_parts, val_state_parts, val_lengths)

    test_dataset, _, _, test_demos = pack_dataset(test_action_parts, test_state_parts, test_lengths)



    generator = torch.Generator().manual_seed(seed)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True, generator=generator)

    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)



    metadata = {

        'dataset_name': POINTMAZE_NAME,

        'display_name': POINTMAZE_LABEL,

        'dataset_id': dataset_id,

        'state_dim': state_dim,

        'action_dim': action_dim,

        'train_demos': train_demos,

        'val_demos': val_demos,

        'test_demos': test_demos,

        'train_windows': len(train_dataset),

        'val_windows': len(val_dataset),

        'test_windows': len(test_dataset),

        'pred_horizon': pred_horizon,

        'obs_horizon': obs_horizon,

        'action_horizon': action_horizon,

        'total_episodes_available': total_episodes,

    }

    return train_loader, val_loader, test_loader, metadata





print(f'✓ PointMaze support ready: {POINTMAZE_DATASET_ID}')


In [ ]:
# ---- Run the RoboMimic Can benchmark with the same JEPA heads ----
if not ROBOMIMIC_CAN_DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Missing RoboMimic Can dataset at {ROBOMIMIC_CAN_DATASET_PATH}. "
        "Download v1.5/can/ph/low_dim_v15.hdf5 before running this cell."
    )

robomimic_train_loader, robomimic_val_loader, robomimic_test_loader, robomimic_meta = build_robomimic_can_loaders(
    dataset_path=ROBOMIMIC_CAN_DATASET_PATH,
    pred_horizon=16,
    obs_horizon=2,
    action_horizon=8,
    batch_size=ROBOMIMIC_BATCH_SIZE,
)

print(f"✓ RoboMimic Can dataset ready: {robomimic_meta['train_windows']} train | {robomimic_meta['val_windows']} val | {robomimic_meta['test_windows']} test windows")
print(f"  Demos — train: {robomimic_meta['train_demos']} | val: {robomimic_meta['val_demos']} | test: {robomimic_meta['test_demos']}")
print(f"  State dim: {robomimic_meta['state_dim']} | Action dim: {robomimic_meta['action_dim']}")

robomimic_can_results, robomimic_can_history = run_dataset_benchmark(
    dataset_name=ROBOMIMIC_CAN_NAME,
    display_name=ROBOMIMIC_CAN_LABEL,
    train_loader=robomimic_train_loader,
    val_loader=robomimic_val_loader,
    test_loader=robomimic_test_loader,
    state_dim=robomimic_meta['state_dim'],
    action_dim=robomimic_meta['action_dim'],
    num_epochs=ROBOMIMIC_NUM_EPOCHS,
    patience=ROBOMIMIC_PATIENCE,
    batch_size=ROBOMIMIC_BATCH_SIZE,
)

print("\n✓ RoboMimic Can benchmark finished")
print(f"  Eval cache: {Path(ARTIFACT_DIR) / f'eval_results_{ROBOMIMIC_CAN_NAME}_{MULTI_DATASET_TAG}.json'}")
print(f"  Training cache: {Path(ARTIFACT_DIR) / f'training_history_{ROBOMIMIC_CAN_NAME}_{MULTI_DATASET_TAG}.json'}")

In [ ]:
# ---- Run the PointMaze benchmark with the same JEPA heads ----

pointmaze_eval_cache = Path(ARTIFACT_DIR) / f"eval_results_{POINTMAZE_NAME}_{MULTI_DATASET_TAG}.json"

pointmaze_history_cache = Path(ARTIFACT_DIR) / f"training_history_{POINTMAZE_NAME}_{MULTI_DATASET_TAG}.json"



pointmaze_train_loader, pointmaze_val_loader, pointmaze_test_loader, pointmaze_meta = build_pointmaze_loaders(

    dataset_id=POINTMAZE_DATASET_ID,

    pred_horizon=16,

    obs_horizon=2,

    action_horizon=8,

    batch_size=POINTMAZE_BATCH_SIZE,

)



print(

    f"✓ PointMaze dataset ready: {pointmaze_meta['train_windows']} train | "

    f"{pointmaze_meta['val_windows']} val | {pointmaze_meta['test_windows']} test windows"

)

print(

    f"  Episodes — train: {pointmaze_meta['train_demos']} | val: {pointmaze_meta['val_demos']} | "

    f"test: {pointmaze_meta['test_demos']} (from {pointmaze_meta['total_episodes_available']} total)"

)

print(f"  State dim: {pointmaze_meta['state_dim']} | Action dim: {pointmaze_meta['action_dim']}")



pointmaze_results, pointmaze_history = run_dataset_benchmark(

    dataset_name=POINTMAZE_NAME,

    display_name=POINTMAZE_LABEL,

    train_loader=pointmaze_train_loader,

    val_loader=pointmaze_val_loader,

    test_loader=pointmaze_test_loader,

    state_dim=pointmaze_meta['state_dim'],

    action_dim=pointmaze_meta['action_dim'],

    num_epochs=POINTMAZE_NUM_EPOCHS,

    patience=POINTMAZE_PATIENCE,

    batch_size=POINTMAZE_BATCH_SIZE,

)



print("\n✓ PointMaze benchmark finished")

print(f"  Eval cache: {pointmaze_eval_cache}")

print(f"  Training cache: {pointmaze_history_cache}")


In [ ]:
# ---- Aggregate Push-T + RoboMimic Can + PointMaze into shared comparison plots ----

PUSHT_COMBINED_CACHE = Path(ARTIFACT_DIR) / f"eval_results_pusht_{MULTI_DATASET_TAG}.json"

ROBOMIMIC_COMBINED_CACHE = Path(ARTIFACT_DIR) / f"eval_results_{ROBOMIMIC_CAN_NAME}_{MULTI_DATASET_TAG}.json"

POINTMAZE_COMBINED_CACHE = Path(ARTIFACT_DIR) / f"eval_results_{POINTMAZE_NAME}_{MULTI_DATASET_TAG}.json"

PUSHT_COMBINED_HISTORY = Path(ARTIFACT_DIR) / f"training_history_pusht_{MULTI_DATASET_TAG}.json"

ROBOMIMIC_COMBINED_HISTORY = Path(ARTIFACT_DIR) / f"training_history_{ROBOMIMIC_CAN_NAME}_{MULTI_DATASET_TAG}.json"

POINTMAZE_COMBINED_HISTORY = Path(ARTIFACT_DIR) / f"training_history_{POINTMAZE_NAME}_{MULTI_DATASET_TAG}.json"

COMBINED_STEM = f"pusht_vs_robomimic_can_vs_pointmaze_{MULTI_DATASET_TAG}"



if not PUSHT_COMBINED_CACHE.exists():

    raise FileNotFoundError(f"Missing Push-T eval cache at {PUSHT_COMBINED_CACHE}")

if not ROBOMIMIC_COMBINED_CACHE.exists():

    raise FileNotFoundError(f"Missing RoboMimic Can eval cache at {ROBOMIMIC_COMBINED_CACHE}")

if not POINTMAZE_COMBINED_CACHE.exists():

    raise FileNotFoundError(f"Missing PointMaze eval cache at {POINTMAZE_COMBINED_CACHE}")



push_cache = load_json(str(PUSHT_COMBINED_CACHE))

robomimic_cache = load_json(str(ROBOMIMIC_COMBINED_CACHE))

pointmaze_cache = load_json(str(POINTMAZE_COMBINED_CACHE))

dataset_payloads = [

    (PUSHT_LABEL, push_cache, load_json(str(PUSHT_COMBINED_HISTORY)) if PUSHT_COMBINED_HISTORY.exists() else None),

    (ROBOMIMIC_CAN_LABEL, robomimic_cache, load_json(str(ROBOMIMIC_COMBINED_HISTORY)) if ROBOMIMIC_COMBINED_HISTORY.exists() else None),

    (POINTMAZE_LABEL, pointmaze_cache, load_json(str(POINTMAZE_COMBINED_HISTORY)) if POINTMAZE_COMBINED_HISTORY.exists() else None),

]



dataset_colors = {

    PUSHT_LABEL: '#1f77b4',

    ROBOMIMIC_CAN_LABEL: '#2ca02c',

    POINTMAZE_LABEL: '#ff7f0e',

}

model_linestyles = {

    'Liquid + MDN': '-',

    'Diffusion': '--',

}

model_markers = {

    'Liquid + MDN': 'o',

    'Diffusion': 's',

}

dataset_title = ', '.join([name for name, _, _ in dataset_payloads[:-1]]) + f", and {dataset_payloads[-1][0]}"



combined_core_rows = []

combined_compute_rows = []

combined_efficiency_rows = []

for dataset_label, payload, _ in dataset_payloads:

    for row in payload['core_table']:

        row_copy = dict(row)

        row_copy['Dataset'] = dataset_label

        combined_core_rows.append(row_copy)

    for row in payload['compute_table']:

        row_copy = dict(row)

        row_copy['Dataset'] = dataset_label

        combined_compute_rows.append(row_copy)

    for row in payload['efficiency_table']:

        row_copy = dict(row)

        row_copy['Dataset'] = dataset_label

        combined_efficiency_rows.append(row_copy)



combined_core_df = pd.DataFrame(combined_core_rows)

combined_compute_df = pd.DataFrame(combined_compute_rows)

combined_efficiency_df = pd.DataFrame(combined_efficiency_rows)

combined_core_df = combined_core_df[['Dataset', 'Model', '# Params', 'NLL ↓', 'MSE ↓', 'Best-of-10 MSE ↓', 'Diversity ↑', 'Smoothness ↓', 'Time (ms) ↓']]

combined_compute_df = combined_compute_df[['Dataset', 'Model', 'Relative Compute x', 'Speedup vs Diffusion ↑', 'Best-of-10 MSE ↓', 'Time (ms) ↓']]

combined_efficiency_df = combined_efficiency_df[['Dataset', 'Model', '# Params', 'Params vs Diffusion x', 'Latency Speedup vs Diffusion ↑', 'NLL ↓', 'Best-of-10 MSE ↓']]

combined_core_df.to_csv(Path(ARTIFACT_DIR) / f"offline_trajectory_modeling_performance_{COMBINED_STEM}.csv", index=False)

combined_compute_df.to_csv(Path(ARTIFACT_DIR) / f"compute_normalized_performance_{COMBINED_STEM}.csv", index=False)

combined_efficiency_df.to_csv(Path(ARTIFACT_DIR) / f"parameter_efficiency_{COMBINED_STEM}.csv", index=False)



# Combined table figure

fig_table, axes = plt.subplots(3, 1, figsize=(22, 14))

for ax, df, title in zip(

    axes,

    [combined_core_df, combined_compute_df, combined_efficiency_df],

    [

        f'Offline Trajectory Modeling Performance: {dataset_title}',

        f'Compute-Normalized Performance: {dataset_title}',

        f'Parameter Efficiency: {dataset_title}',

    ],

):

    ax.axis('off')

    display_df = df.copy()

    for col in display_df.columns:

        if col not in ['Dataset', 'Model']:

            display_df[col] = display_df[col].map(lambda x: '—' if pd.isna(x) else f'{x:.4f}' if isinstance(x, (int, float, np.floating)) else x)

    table = ax.table(cellText=display_df.values, colLabels=display_df.columns, loc='center', cellLoc='center')

    table.auto_set_font_size(False)

    table.set_fontsize(8)

    table.scale(1.05, 1.35)

    ax.set_title(title, fontsize=12, fontweight='bold')

plt.tight_layout()

plt.savefig(Path(ARTIFACT_DIR) / f"offline_tables_{COMBINED_STEM}.png", dpi=150, bbox_inches='tight')

plt.show()



# Best-of-K comparison

fig1, ax = plt.subplots(1, 1, figsize=(9.5, 5.5))

for dataset_label, payload, _ in dataset_payloads:

    color = dataset_colors[dataset_label]

    k_values = [int(k) for k in payload['liquid_results']['k_metrics'].keys()]

    k_values.sort()

    liquid_best = [payload['liquid_results']['k_metrics'][str(k)]['best_of_k_mse'] for k in k_values]

    diffusion_best = [payload['diffusion_results']['k_metrics'][str(k)]['best_of_k_mse'] for k in k_values]

    ax.plot(k_values, liquid_best, color=color, linestyle='-', marker='o', linewidth=2)

    ax.plot(k_values, diffusion_best, color=color, linestyle='--', marker='s', linewidth=2)

ax.set_yscale('log')

ax.set_xlabel('Number of samples K')

ax.set_ylabel('Best-of-K MSE (log scale)')

ax.set_title(f'Best-of-K vs K across {dataset_title}')

ax.grid(True, alpha=0.3, which='both')

dataset_handles = [Line2D([0], [0], color=dataset_colors[name], lw=2, label=name) for name, _, _ in dataset_payloads]

model_handles = [

    Line2D([0], [0], color='black', lw=2, linestyle='-', marker='o', label='Liquid + MDN'),

    Line2D([0], [0], color='black', lw=2, linestyle='--', marker='s', label='Diffusion'),

]

legend1 = ax.legend(handles=dataset_handles, title='Dataset', loc='upper right')

ax.add_artist(legend1)

ax.legend(handles=model_handles, title='Model', loc='lower left')

plt.tight_layout()

plt.savefig(Path(ARTIFACT_DIR) / f"best_of_k_curve_{COMBINED_STEM}.png", dpi=150, bbox_inches='tight')

plt.show()



# Qualitative comparison using the first two action dimensions and three examples per model

qual_n = min(

    NUM_QUAL_EXAMPLES,

    *[len(payload['liquid_results']['qualitative']['target']) for _, payload, _ in dataset_payloads],

    *[len(payload['diffusion_results']['qualitative']['target']) for _, payload, _ in dataset_payloads],

)

fig2, axes = plt.subplots(len(dataset_payloads), qual_n * 2, figsize=(5.1 * qual_n * 2, 4.4 * len(dataset_payloads)), squeeze=False)

for row_idx, (dataset_label, payload, _) in enumerate(dataset_payloads):

    for ex_idx in range(qual_n):

        gt = np.asarray(payload['liquid_results']['qualitative']['target'][ex_idx])[:, :2]

        liquid_det = np.asarray(payload['liquid_results']['qualitative']['deterministic'][ex_idx])[:, :2]

        liquid_samples = np.asarray(payload['liquid_results']['qualitative']['samples'][ex_idx])[:, :, :2]

        diffusion_proxy = np.asarray(payload['diffusion_results']['qualitative']['proxy_mean'][ex_idx])[:, :2]

        diffusion_samples = np.asarray(payload['diffusion_results']['qualitative']['samples'][ex_idx])[:, :, :2]

        combined = np.concatenate([

            gt,

            liquid_det,

            liquid_samples.reshape(-1, 2),

            diffusion_proxy,

            diffusion_samples.reshape(-1, 2),

        ], axis=0)

        xmin, ymin = combined.min(axis=0)

        xmax, ymax = combined.max(axis=0)

        xpad = max(0.05, 0.1 * float(xmax - xmin))

        ypad = max(0.05, 0.1 * float(ymax - ymin))



        ax_l = axes[row_idx, 2 * ex_idx]

        for sample in liquid_samples:

            ax_l.plot(sample[:, 0], sample[:, 1], color=dataset_colors[dataset_label], alpha=0.18)

        ax_l.plot(liquid_det[:, 0], liquid_det[:, 1], color=dataset_colors[dataset_label], linestyle='--', linewidth=2, label=f'{dataset_label} liquid mean')

        ax_l.plot(gt[:, 0], gt[:, 1], color='black', linewidth=2.5, label='Ground truth')

        ax_l.set_xlim(xmin - xpad, xmax + xpad)

        ax_l.set_ylim(ymin - ypad, ymax + ypad)

        ax_l.set_title(f'{dataset_label}: Liquid example {ex_idx + 1}')

        ax_l.set_xlabel('action[0]')

        if ex_idx == 0:

            ax_l.set_ylabel('action[1]')

        ax_l.grid(True, alpha=0.3)

        if ex_idx == 0:

            ax_l.legend(loc='best')



        ax_d = axes[row_idx, 2 * ex_idx + 1]

        for sample in diffusion_samples:

            ax_d.plot(sample[:, 0], sample[:, 1], color=dataset_colors[dataset_label], alpha=0.18)

        ax_d.plot(diffusion_proxy[:, 0], diffusion_proxy[:, 1], color=dataset_colors[dataset_label], linestyle='--', linewidth=2, label=f'{dataset_label} diffusion proxy mean')

        ax_d.plot(gt[:, 0], gt[:, 1], color='black', linewidth=2.5, label='Ground truth')

        ax_d.set_xlim(xmin - xpad, xmax + xpad)

        ax_d.set_ylim(ymin - ypad, ymax + ypad)

        ax_d.set_title(f'{dataset_label}: Diffusion example {ex_idx + 1}')

        ax_d.set_xlabel('action[0]')

        ax_d.grid(True, alpha=0.3)

        if ex_idx == 0:

            ax_d.legend(loc='best')

plt.tight_layout()

plt.savefig(Path(ARTIFACT_DIR) / f"qualitative_trajectory_samples_{COMBINED_STEM}.png", dpi=150, bbox_inches='tight')

plt.show()



# Error vs horizon

fig3, ax = plt.subplots(1, 1, figsize=(9.5, 5.5))

for dataset_label, payload, _ in dataset_payloads:

    color = dataset_colors[dataset_label]

    horizon = np.arange(1, len(payload['liquid_results']['deterministic']['per_horizon_mse']) + 1)

    ax.plot(horizon, payload['liquid_results']['k_metrics']['10']['per_horizon_best_of_k_mse'], color=color, linestyle='-', linewidth=2)

    ax.plot(horizon, payload['diffusion_results']['k_metrics']['10']['per_horizon_best_of_k_mse'], color=color, linestyle='--', linewidth=2)

ax.set_yscale('log')

ax.set_xlabel('Prediction horizon step')

ax.set_ylabel('Best-of-10 MSE (log scale)')

ax.set_title(f'Error vs Horizon across {dataset_title}')

ax.grid(True, alpha=0.3, which='both')

legend1 = ax.legend(handles=dataset_handles, title='Dataset', loc='upper left')

ax.add_artist(legend1)

ax.legend(handles=model_handles, title='Model', loc='lower right')

plt.tight_layout()

plt.savefig(Path(ARTIFACT_DIR) / f"error_vs_horizon_{COMBINED_STEM}.png", dpi=150, bbox_inches='tight')

plt.show()



# Diversity vs error

fig4, ax = plt.subplots(1, 1, figsize=(8.8, 5.8))

marker_map = {'Liquid + MDN': 'o', 'Diffusion': '^'}

for dataset_label, payload, _ in dataset_payloads:

    color = dataset_colors[dataset_label]

    ax.scatter(

        np.asarray(payload['liquid_results']['k_metrics']['10']['per_example_diversity']) + 1e-6,

        np.asarray(payload['liquid_results']['k_metrics']['10']['per_example_best_of_k_mse']) + 1e-6,

        color=color,

        alpha=0.28,

        s=18,

        marker=marker_map['Liquid + MDN'],

    )

    ax.scatter(

        np.asarray(payload['diffusion_results']['k_metrics']['10']['per_example_diversity']) + 1e-6,

        np.asarray(payload['diffusion_results']['k_metrics']['10']['per_example_best_of_k_mse']) + 1e-6,

        color=color,

        alpha=0.28,

        s=18,

        marker=marker_map['Diffusion'],

    )

ax.set_xscale('log')

ax.set_yscale('log')

ax.set_xlabel('Diversity (log scale)')

ax.set_ylabel('Best-of-10 MSE (log scale)')

ax.set_title(f'Diversity vs Error Tradeoff across {dataset_title}')

ax.grid(True, alpha=0.3, which='both')

legend1 = ax.legend(handles=dataset_handles, title='Dataset', loc='upper left')

ax.add_artist(legend1)

marker_handles = [

    Line2D([0], [0], marker='o', color='black', linestyle='None', label='Liquid + MDN'),

    Line2D([0], [0], marker='^', color='black', linestyle='None', label='Diffusion'),

]

ax.legend(handles=marker_handles, title='Model', loc='lower right')

plt.tight_layout()

plt.savefig(Path(ARTIFACT_DIR) / f"diversity_vs_error_{COMBINED_STEM}.png", dpi=150, bbox_inches='tight')

plt.show()



# Training progress comparison (validation curves only for readability)

fig5, ax = plt.subplots(1, 1, figsize=(10, 5.2))

for dataset_label, _, history in dataset_payloads:

    if history is None or len(history.get('epochs', [])) == 0:

        continue

    epochs = history['epochs']

    color = dataset_colors[dataset_label]

    ax.plot(epochs, history['liquid_val_nll'], color=color, linestyle='-', linewidth=2)

    ax.plot(epochs, history['diffusion_val_nll'], color=color, linestyle='--', linewidth=2)

ax.set_xlabel('Epoch')

ax.set_ylabel('Validation NLL / proxy NLL')

ax.set_title(f'Training Progress across {dataset_title}')

ax.grid(True, alpha=0.3)

legend1 = ax.legend(handles=dataset_handles, title='Dataset', loc='upper right')

ax.add_artist(legend1)

ax.legend(handles=model_handles, title='Model', loc='lower left')

plt.tight_layout()

plt.savefig(Path(ARTIFACT_DIR) / f"training_progress_{COMBINED_STEM}.png", dpi=150, bbox_inches='tight')

plt.show()



print('✓ Cross-dataset comparison artifacts saved')

print(f'  - {Path(ARTIFACT_DIR) / f"offline_trajectory_modeling_performance_{COMBINED_STEM}.csv"}')

print(f'  - {Path(ARTIFACT_DIR) / f"compute_normalized_performance_{COMBINED_STEM}.csv"}')

print(f'  - {Path(ARTIFACT_DIR) / f"parameter_efficiency_{COMBINED_STEM}.csv"}')

print(f'  - {Path(ARTIFACT_DIR) / f"offline_tables_{COMBINED_STEM}.png"}')

print(f'  - {Path(ARTIFACT_DIR) / f"best_of_k_curve_{COMBINED_STEM}.png"}')

print(f'  - {Path(ARTIFACT_DIR) / f"qualitative_trajectory_samples_{COMBINED_STEM}.png"}')

print(f'  - {Path(ARTIFACT_DIR) / f"error_vs_horizon_{COMBINED_STEM}.png"}')

print(f'  - {Path(ARTIFACT_DIR) / f"diversity_vs_error_{COMBINED_STEM}.png"}')

print(f'  - {Path(ARTIFACT_DIR) / f"training_progress_{COMBINED_STEM}.png"}')


## PointMaze Gymnasium Evaluation + Episode GIF

Evaluate the PointMaze JEPA checkpoints in a Gymnasium environment and render a side-by-side animated GIF (Liquid vs Diffusion).

In [ ]:
# PointMaze checkpoint load + policy adapters

import os
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F

PM_OBS_WINDOW = 2
PM_STATE_DIM = 8
PM_STATE_KEYS = ["observation", "achieved_goal", "desired_goal"]

checkpoint_dir = Path(os.getcwd()) / "checkpoints"
liquid_pm_ckpt = checkpoint_dir / "liquid_jepa_pointmaze_umaze_fair_halfparam_deterministic_clip_120epochs_best.pt"
diffusion_pm_ckpt = checkpoint_dir / "diffusion_jepa_pointmaze_umaze_fair_halfparam_deterministic_clip_120epochs_best.pt"

assert liquid_pm_ckpt.exists(), f"Missing checkpoint: {liquid_pm_ckpt}"
assert diffusion_pm_ckpt.exists(), f"Missing checkpoint: {diffusion_pm_ckpt}"

pm_device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
)

liquid_pm_state = torch.load(liquid_pm_ckpt, map_location=pm_device, weights_only=False)
diffusion_pm_state = torch.load(diffusion_pm_ckpt, map_location=pm_device, weights_only=False)

# Liquid tensors
liq_state_proj_w = liquid_pm_state["backbone.state_proj.weight"]
liq_state_proj_b = liquid_pm_state["backbone.state_proj.bias"]
liq_context_proj_w = liquid_pm_state["backbone.context_proj.weight"]
liq_context_proj_b = liquid_pm_state["backbone.context_proj.bias"]
liq_head_context_w = liquid_pm_state["context_proj.weight"]
liq_head_context_b = liquid_pm_state["context_proj.bias"]
liq_mdn_logits_w = liquid_pm_state["mdn_logits.weight"]
liq_mdn_logits_b = liquid_pm_state["mdn_logits.bias"]
liq_mdn_mu_w = liquid_pm_state["mdn_mu.weight"]
liq_mdn_mu_b = liquid_pm_state["mdn_mu.bias"]

# Diffusion tensors
diff_state_proj_w = diffusion_pm_state["backbone.state_proj.weight"]
diff_state_proj_b = diffusion_pm_state["backbone.state_proj.bias"]
diff_context_proj_w = diffusion_pm_state["backbone.context_proj.weight"]
diff_context_proj_b = diffusion_pm_state["backbone.context_proj.bias"]
diff_head_context_w = diffusion_pm_state["context_proj.weight"]
diff_head_context_b = diffusion_pm_state["context_proj.bias"]
diff_mdn_logits_w = diffusion_pm_state["mdn_logits.weight"]
diff_mdn_logits_b = diffusion_pm_state["mdn_logits.bias"]
diff_mdn_mu_w = diffusion_pm_state["mdn_mu.weight"]
diff_mdn_mu_b = diffusion_pm_state["mdn_mu.bias"]


def _pm_backbone_context(obs_history_np, state_proj_w, state_proj_b, backbone_context_w, backbone_context_b):
    x = torch.as_tensor(obs_history_np, dtype=torch.float32, device=pm_device).unsqueeze(0)
    tok = torch.tanh(torch.matmul(x, state_proj_w.t()) + state_proj_b)
    ctx = torch.tanh(torch.matmul(tok.mean(dim=1), backbone_context_w.t()) + backbone_context_b)
    return ctx


def _pm_action_from_context(ctx, head_context_w, head_context_b, mdn_logits_w, mdn_logits_b, mdn_mu_w, mdn_mu_b):
    h = torch.tanh(torch.matmul(ctx, head_context_w.t()) + head_context_b)
    logits = torch.clamp(torch.matmul(h, mdn_logits_w.t()) + mdn_logits_b, -30.0, 30.0)
    mu = torch.matmul(h, mdn_mu_w.t()) + mdn_mu_b
    K = logits.shape[-1]
    mu = mu.view(1, K, 2)
    best_k = torch.argmax(F.log_softmax(logits, dim=-1), dim=-1)
    action = mu[0, best_k.item()]
    return action.detach().cpu().numpy()


def liquid_pointmaze_action(obs_history_np):
    ctx = _pm_backbone_context(
        obs_history_np,
        liq_state_proj_w, liq_state_proj_b,
        liq_context_proj_w, liq_context_proj_b,
    )
    return _pm_action_from_context(
        ctx,
        liq_head_context_w, liq_head_context_b,
        liq_mdn_logits_w, liq_mdn_logits_b,
        liq_mdn_mu_w, liq_mdn_mu_b,
    )


def diffusion_pointmaze_action(obs_history_np):
    ctx = _pm_backbone_context(
        obs_history_np,
        diff_state_proj_w, diff_state_proj_b,
        diff_context_proj_w, diff_context_proj_b,
    )
    return _pm_action_from_context(
        ctx,
        diff_head_context_w, diff_head_context_b,
        diff_mdn_logits_w, diff_mdn_logits_b,
        diff_mdn_mu_w, diff_mdn_mu_b,
    )


_dummy = np.zeros((PM_OBS_WINDOW, PM_STATE_DIM), dtype=np.float32)
print("Liquid action sample:", np.round(liquid_pointmaze_action(_dummy), 4))
print("Diffusion action sample:", np.round(diffusion_pointmaze_action(_dummy), 4))
print(f"✓ PointMaze adapters ready on {pm_device}")

In [ ]:
# Continue PointMaze training for +120 epochs using the SAME JEPA models/objectives with stable continuation settings

import re
import time
import copy
from pathlib import Path
from tqdm.auto import tqdm

PM_CONT_EXTRA_EPOCHS = 120
PM_CONT_PATIENCE = 999
PM_CONT_BATCH_SIZE = POINTMAZE_BATCH_SIZE
PM_CONT_MIN_DELTA = MIN_DELTA
PM_CONT_DEVICE = device

# Keep end-of-training regime from epoch 120 onward
PM_CONT_TEACHER_FORCING = 0.15
PM_CONT_FREE_W = 0.75
PM_CONT_TF_W = 1.0 - PM_CONT_FREE_W

# LR floor for meaningful continuation updates
PM_CONT_RESUME_LR_MIN = 3e-6

# Always reload clean 120-epoch checkpoints from disk at cell start
liquid_pm_state = torch.load(liquid_pm_ckpt, map_location=PM_CONT_DEVICE, weights_only=False)
diffusion_pm_state = torch.load(diffusion_pm_ckpt, map_location=PM_CONT_DEVICE, weights_only=False)


def _infer_epoch_from_name(path_obj):
    m = re.search(r"(\d+)epochs", str(path_obj))
    return int(m.group(1)) if m else 120


start_epoch = max(_infer_epoch_from_name(liquid_pm_ckpt), _infer_epoch_from_name(diffusion_pm_ckpt))
end_epoch = start_epoch + PM_CONT_EXTRA_EPOCHS

# Same data protocol as prior PointMaze benchmark
pointmaze_train_loader, pointmaze_val_loader, pointmaze_test_loader, pointmaze_meta = build_pointmaze_loaders(
    dataset_id=POINTMAZE_DATASET_ID,
    pred_horizon=16,
    obs_horizon=2,
    action_horizon=8,
    batch_size=PM_CONT_BATCH_SIZE,
)

state_dim = pointmaze_meta['state_dim']
action_dim = pointmaze_meta['action_dim']

# Rebuild exact models
cont_backbone_liq = SharedBackbone(
    clip_model=clip_model,
    clip_feat_dim=CLIP_FEAT_DIM,
    state_dim=state_dim,
    hidden_dim=D_MODEL,
    num_layers=4,
).to(PM_CONT_DEVICE)

cont_backbone_diff = SharedBackbone(
    clip_model=clip_model,
    clip_feat_dim=CLIP_FEAT_DIM,
    state_dim=state_dim,
    hidden_dim=D_MODEL,
    num_layers=4,
).to(PM_CONT_DEVICE)

liquid_model_pm = LiquidTrajectoryModel(
    backbone=cont_backbone_liq,
    action_dim=action_dim,
    hidden_dim=LIQUID_HIDDEN_SIZE,
    seq_length=16,
    num_layers=NUM_LAYERS,
    num_mixtures=NUM_MIXTURES,
    dropout=DROPOUT,
).to(PM_CONT_DEVICE)

diffusion_model_pm = DiffusionTrajectoryModel(
    backbone=cont_backbone_diff,
    action_dim=action_dim,
    hidden_dim=DIFFUSION_HIDDEN_SIZE,
    seq_length=16,
    num_diffusion_steps=NUM_DIFFUSION_STEPS,
    num_mixtures=NUM_MIXTURES,
).to(PM_CONT_DEVICE)

liquid_model_pm.load_state_dict(liquid_pm_state, strict=True)
diffusion_model_pm.load_state_dict(diffusion_pm_state, strict=True)

# Use last known LRs from the original 120-epoch PointMaze training cache (if present)
pointmaze_history_cache = Path(ARTIFACT_DIR) / f"training_history_{POINTMAZE_NAME}_{MULTI_DATASET_TAG}.json"
resume_liq_lr = LR * 0.05
resume_diff_lr = LR * 0.05
if pointmaze_history_cache.exists():
    hist = load_json(str(pointmaze_history_cache))
    if isinstance(hist, dict):
        liq_lrs = hist.get('liquid_lr', [])
        diff_lrs = hist.get('diffusion_lr', [])
        if isinstance(liq_lrs, list) and len(liq_lrs) > 0:
            resume_liq_lr = float(liq_lrs[-1])
        if isinstance(diff_lrs, list) and len(diff_lrs) > 0:
            resume_diff_lr = float(diff_lrs[-1])

# Raise tiny tail LRs to a useful floor for continued optimization
resume_liq_lr = max(resume_liq_lr, PM_CONT_RESUME_LR_MIN)
resume_diff_lr = max(resume_diff_lr, PM_CONT_RESUME_LR_MIN)

opt_liq = torch.optim.AdamW(liquid_model_pm.parameters(), lr=resume_liq_lr, weight_decay=WEIGHT_DECAY)
opt_diff = torch.optim.AdamW(diffusion_model_pm.parameters(), lr=resume_diff_lr, weight_decay=WEIGHT_DECAY)

# Gentle tail schedule for continuation only
sch_liq = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_liq,
    T_max=max(1, PM_CONT_EXTRA_EPOCHS),
    eta_min=max(1e-6, resume_liq_lr * 0.2),
)
sch_diff = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_diff,
    T_max=max(1, PM_CONT_EXTRA_EPOCHS),
    eta_min=max(1e-6, resume_diff_lr * 0.2),
)


def _eval_val_metrics(liquid_model_local, diffusion_model_local, val_loader):
    liquid_model_local.eval()
    diffusion_model_local.eval()

    liquid_val_nll = 0.0
    diffusion_val_nll = 0.0
    diffusion_val_ddpm = 0.0
    diffusion_val_total = 0.0

    with torch.no_grad():
        for batch in val_loader:
            images = batch['image'].to(PM_CONT_DEVICE)
            state = batch['state'].to(PM_CONT_DEVICE)
            action_target = batch['action'].to(PM_CONT_DEVICE)

            out_liquid = liquid_model_local(images, state, teacher_forcing_ratio=0.0, return_mdn=True)
            liquid_val_nll += mdn_nll_loss(out_liquid['logits'], out_liquid['mu'], out_liquid['log_sigma'], action_target).item()

            out_diffusion = diffusion_model_local(images, state, action_target=action_target, return_mdn=True, inference=False)
            diff_val_nll = mdn_nll_loss(out_diffusion['logits'], out_diffusion['mu'], out_diffusion['log_sigma'], action_target)
            diff_val_ddpm = F.mse_loss(out_diffusion['pred_epsilon'], out_diffusion['target_epsilon'])

            diffusion_val_nll += diff_val_nll.item()
            diffusion_val_ddpm += diff_val_ddpm.item()
            diffusion_val_total += (DIFFUSION_NLL_WEIGHT * diff_val_nll + DIFFUSION_DDPM_WEIGHT * diff_val_ddpm).item()

    n = max(1, len(val_loader))
    return (
        liquid_val_nll / n,
        diffusion_val_nll / n,
        diffusion_val_ddpm / n,
        diffusion_val_total / n,
    )


def _preview_train_loss(liquid_model_local, diffusion_model_local, train_loader, num_batches=20):
    liquid_model_local.eval()
    diffusion_model_local.eval()

    liq_total = 0.0
    diff_total = 0.0
    n = 0
    with torch.no_grad():
        for batch in train_loader:
            images = batch['image'].to(PM_CONT_DEVICE)
            state = batch['state'].to(PM_CONT_DEVICE)
            action_target = batch['action'].to(PM_CONT_DEVICE)

            out_tf = liquid_model_local(images, state, action_target=action_target, teacher_forcing_ratio=PM_CONT_TEACHER_FORCING, return_mdn=True)
            loss_tf = mdn_nll_loss(out_tf['logits'], out_tf['mu'], out_tf['log_sigma'], action_target)
            out_free = liquid_model_local(images, state, action_target=None, teacher_forcing_ratio=0.0, return_mdn=True)
            loss_free = mdn_nll_loss(out_free['logits'], out_free['mu'], out_free['log_sigma'], action_target)
            liq_loss = PM_CONT_TF_W * loss_tf + PM_CONT_FREE_W * loss_free

            out_diff = diffusion_model_local(images, state, action_target=action_target, return_mdn=True, inference=False)
            loss_nll = mdn_nll_loss(out_diff['logits'], out_diff['mu'], out_diff['log_sigma'], action_target)
            loss_ddpm = F.mse_loss(out_diff['pred_epsilon'], out_diff['target_epsilon'])
            diff_loss = DIFFUSION_NLL_WEIGHT * loss_nll + DIFFUSION_DDPM_WEIGHT * loss_ddpm

            liq_total += float(liq_loss.item())
            diff_total += float(diff_loss.item())
            n += 1
            if n >= num_batches:
                break

    return liq_total / max(1, n), diff_total / max(1, n)


base_liq_val, base_diff_nll, base_diff_ddpm, base_diff_total = _eval_val_metrics(
    liquid_model_pm, diffusion_model_pm, pointmaze_val_loader
)
preview_liq_train, preview_diff_train = _preview_train_loss(liquid_model_pm, diffusion_model_pm, pointmaze_train_loader)

print(f"Resuming same PointMaze JEPA models from epoch {start_epoch} to {end_epoch}")
print(f"Resumed LR | Liquid: {opt_liq.param_groups[0]['lr']:.3e} | Diffusion: {opt_diff.param_groups[0]['lr']:.3e}")
print(f"LR floor applied: {PM_CONT_RESUME_LR_MIN:.1e}")
print(f"Baseline val NLL | Liquid: {base_liq_val:.4f} | Diffusion: {base_diff_nll:.4f}")
print(f"Baseline val total (Diffusion): {base_diff_total:.4f} (NLL weight={DIFFUSION_NLL_WEIGHT}, DDPM weight={DIFFUSION_DDPM_WEIGHT})")
print(f"Preview train objective before updates | Liquid: {preview_liq_train:.4f} | Diffusion total: {preview_diff_train:.4f}")

best_liq = base_liq_val
best_diff_total = base_diff_total
best_liq_sd = copy.deepcopy(liquid_model_pm.state_dict())
best_diff_sd = copy.deepcopy(diffusion_model_pm.state_dict())
liquid_no_improve = 0
diff_no_improve = 0

for epoch in range(start_epoch, end_epoch):
    epoch_start = time.perf_counter()

    teacher_forcing_ratio = PM_CONT_TEACHER_FORCING
    free_w = PM_CONT_FREE_W
    tf_w = PM_CONT_TF_W

    liquid_model_pm.train()
    diffusion_model_pm.train()

    liquid_train_total = 0.0
    diffusion_train_nll = 0.0
    diffusion_train_ddpm = 0.0
    diffusion_train_total = 0.0

    with tqdm(pointmaze_train_loader, desc=f"[PointMaze continue] Epoch {epoch+1}/{end_epoch}", leave=False) as pbar:
        for batch in pbar:
            images = batch['image'].to(PM_CONT_DEVICE)
            state = batch['state'].to(PM_CONT_DEVICE)
            action_target = batch['action'].to(PM_CONT_DEVICE)

            opt_liq.zero_grad(set_to_none=True)
            out_tf = liquid_model_pm(images, state, action_target=action_target, teacher_forcing_ratio=teacher_forcing_ratio, return_mdn=True)
            loss_tf = mdn_nll_loss(out_tf['logits'], out_tf['mu'], out_tf['log_sigma'], action_target)
            out_free = liquid_model_pm(images, state, action_target=None, teacher_forcing_ratio=0.0, return_mdn=True)
            loss_free = mdn_nll_loss(out_free['logits'], out_free['mu'], out_free['log_sigma'], action_target)
            liquid_loss = tf_w * loss_tf + free_w * loss_free
            liquid_loss.backward()
            torch.nn.utils.clip_grad_norm_(liquid_model_pm.parameters(), 1.0)
            opt_liq.step()
            liquid_train_total += liquid_loss.item()

            opt_diff.zero_grad(set_to_none=True)
            out_diff = diffusion_model_pm(images, state, action_target=action_target, return_mdn=True, inference=False)
            loss_nll = mdn_nll_loss(out_diff['logits'], out_diff['mu'], out_diff['log_sigma'], action_target)
            loss_ddpm = F.mse_loss(out_diff['pred_epsilon'], out_diff['target_epsilon'])
            diffusion_loss = DIFFUSION_NLL_WEIGHT * loss_nll + DIFFUSION_DDPM_WEIGHT * loss_ddpm
            diffusion_loss.backward()
            torch.nn.utils.clip_grad_norm_(diffusion_model_pm.parameters(), 1.0)
            opt_diff.step()

            diffusion_train_nll += loss_nll.item()
            diffusion_train_ddpm += loss_ddpm.item()
            diffusion_train_total += diffusion_loss.item()

            pbar.set_postfix(liq=f"{liquid_loss.item():.4f}", diff=f"{diffusion_loss.item():.4f}")

    ntr = max(1, len(pointmaze_train_loader))
    avg_liq_train = liquid_train_total / ntr
    avg_diff_train_nll = diffusion_train_nll / ntr
    avg_diff_train_ddpm = diffusion_train_ddpm / ntr
    avg_diff_train_total = diffusion_train_total / ntr

    liq_val, diff_val_nll, diff_val_ddpm, diff_val_total = _eval_val_metrics(
        liquid_model_pm, diffusion_model_pm, pointmaze_val_loader
    )

    sch_liq.step()
    sch_diff.step()

    liq_tag = ""
    diff_tag = ""
    if liq_val < (best_liq - PM_CONT_MIN_DELTA):
        best_liq = liq_val
        best_liq_sd = copy.deepcopy(liquid_model_pm.state_dict())
        liquid_no_improve = 0
        liq_tag = " [best]"
    else:
        liquid_no_improve += 1

    if diff_val_total < (best_diff_total - PM_CONT_MIN_DELTA):
        best_diff_total = diff_val_total
        best_diff_sd = copy.deepcopy(diffusion_model_pm.state_dict())
        diff_no_improve = 0
        diff_tag = " [best]"
    else:
        diff_no_improve += 1

    epoch_time = time.perf_counter() - epoch_start
    print(f"Epoch {epoch+1}/{end_epoch} | Time: {epoch_time:.1f}s")
    print(f"  [Liquid]    Train NLL: {avg_liq_train:.5f} | Val NLL: {liq_val:.5f}{liq_tag}")
    print(f"  [Diffusion] Train total: {avg_diff_train_total:.5f} | Val total: {diff_val_total:.5f}{diff_tag}")
    print(f"               Train NLL: {avg_diff_train_nll:.5f} | Val NLL: {diff_val_nll:.5f}")
    print(f"               Train DDPM: {avg_diff_train_ddpm:.5f} | Val DDPM: {diff_val_ddpm:.5f}")
    print(f"               LR(liq/diff): {opt_liq.param_groups[0]['lr']:.3e}/{opt_diff.param_groups[0]['lr']:.3e}")

    if liquid_no_improve > PM_CONT_PATIENCE and diff_no_improve > PM_CONT_PATIENCE:
        print("Early stop triggered by patience for both models.")
        break

# restore best continuation states
liquid_model_pm.load_state_dict(best_liq_sd)
diffusion_model_pm.load_state_dict(best_diff_sd)

# refresh checkpoint dicts for downstream policy adapter evaluation cells
liquid_pm_state = copy.deepcopy(liquid_model_pm.state_dict())
diffusion_pm_state = copy.deepcopy(diffusion_model_pm.state_dict())

# refresh adapter tensor references used by evaluation/GIF cells
liq_state_proj_w = liquid_pm_state["backbone.state_proj.weight"].to(pm_device)
liq_state_proj_b = liquid_pm_state["backbone.state_proj.bias"].to(pm_device)
liq_context_proj_w = liquid_pm_state["backbone.context_proj.weight"].to(pm_device)
liq_context_proj_b = liquid_pm_state["backbone.context_proj.bias"].to(pm_device)
liq_head_context_w = liquid_pm_state["context_proj.weight"].to(pm_device)
liq_head_context_b = liquid_pm_state["context_proj.bias"].to(pm_device)
liq_mdn_logits_w = liquid_pm_state["mdn_logits.weight"].to(pm_device)
liq_mdn_logits_b = liquid_pm_state["mdn_logits.bias"].to(pm_device)
liq_mdn_mu_w = liquid_pm_state["mdn_mu.weight"].to(pm_device)
liq_mdn_mu_b = liquid_pm_state["mdn_mu.bias"].to(pm_device)

diff_state_proj_w = diffusion_pm_state["backbone.state_proj.weight"].to(pm_device)
diff_state_proj_b = diffusion_pm_state["backbone.state_proj.bias"].to(pm_device)
diff_context_proj_w = diffusion_pm_state["backbone.context_proj.weight"].to(pm_device)
diff_context_proj_b = diffusion_pm_state["backbone.context_proj.bias"].to(pm_device)
diff_head_context_w = diffusion_pm_state["context_proj.weight"].to(pm_device)
diff_head_context_b = diffusion_pm_state["context_proj.bias"].to(pm_device)
diff_mdn_logits_w = diffusion_pm_state["mdn_logits.weight"].to(pm_device)
diff_mdn_logits_b = diffusion_pm_state["mdn_logits.bias"].to(pm_device)
diff_mdn_mu_w = diffusion_pm_state["mdn_mu.weight"].to(pm_device)
diff_mdn_mu_b = diffusion_pm_state["mdn_mu.bias"].to(pm_device)

# save continued checkpoints
liquid_pm_continued_ckpt = checkpoint_dir / f"liquid_jepa_pointmaze_umaze_fair_halfparam_deterministic_clip_{end_epoch}epochs_continued.pt"
diffusion_pm_continued_ckpt = checkpoint_dir / f"diffusion_jepa_pointmaze_umaze_fair_halfparam_deterministic_clip_{end_epoch}epochs_continued.pt"
torch.save(liquid_pm_state, liquid_pm_continued_ckpt)
torch.save(diffusion_pm_state, diffusion_pm_continued_ckpt)

print("\n✓ Continuation complete with original PointMaze JEPA setup")
print(f"Best continuation val NLL (Liquid): {best_liq:.5f}")
print(f"Best continuation val total (Diffusion): {best_diff_total:.5f}")
print(f"Saved: {liquid_pm_continued_ckpt}")
print(f"Saved: {diffusion_pm_continued_ckpt}")

In [ ]:
# Register Gymnasium Robotics environments (required for official PointMaze IDs)
import gymnasium as gym

try:
    import gymnasium_robotics  # noqa: F401
    pm_ids = sorted([spec.id for spec in gym.envs.registry.values() if "PointMaze" in spec.id])
    print(f"Registered PointMaze env IDs ({len(pm_ids)}):")
    for env_id in pm_ids[:20]:
        print(" -", env_id)
    if len(pm_ids) > 20:
        print(f"... and {len(pm_ids)-20} more")
except Exception as e:
    print("Gymnasium Robotics is not fully available in this environment.")
    print("Reason:", repr(e))
    print("Tip: official PointMaze usually needs Python 3.10-3.12 + MuJoCo runtime.")

In [ ]:
# PointMaze Gymnasium evaluation

import collections
import time
import gymnasium as gym
from gymnasium import spaces
from tqdm.auto import tqdm


def make_pointmaze_env():
    candidates = [
        "PointMaze_UMazeDense-v3",
        "PointMaze_UMaze-v3",
        "PointMaze_UMazeDense-v2",
        "PointMaze_UMaze-v2",
        "PointMaze_UMaze-v4",
    ]

    for env_id in candidates:
        try:
            env = gym.make(env_id)
            print(f"Using env: {env_id}")
            return env, env_id
        except Exception:
            pass

    class PointMazeLiteEnv(gym.Env):
        metadata = {"render_modes": []}

        def __init__(self, max_steps=300):
            super().__init__()
            self.max_steps = max_steps
            self.observation_space = spaces.Box(low=-2.0, high=2.0, shape=(8,), dtype=np.float32)
            self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(2,), dtype=np.float32)
            self.step_count = 0
            self.pos = None
            self.vel = None
            self.goal = np.array([0.75, 0.75], dtype=np.float32)

        def _obs(self):
            d = self.goal - self.pos
            return np.array([
                self.pos[0], self.pos[1],
                self.vel[0], self.vel[1],
                self.goal[0], self.goal[1],
                d[0], d[1],
            ], dtype=np.float32)

        def reset(self, seed=None, options=None):
            super().reset(seed=seed)
            self.step_count = 0
            self.pos = np.array([-0.8, -0.8], dtype=np.float32)
            self.vel = np.zeros(2, dtype=np.float32)
            return self._obs(), {"success": False}

        def _blocked(self, p):
            x, y = p
            in_left_wall = (-0.2 <= x <= -0.1) and (-0.8 <= y <= 0.4)
            in_right_wall = (0.1 <= x <= 0.2) and (-0.8 <= y <= 0.4)
            in_bottom_wall = (-0.2 <= x <= 0.2) and (-0.8 <= y <= -0.7)
            return in_left_wall or in_right_wall or in_bottom_wall

        def step(self, action):
            self.step_count += 1
            a = np.clip(np.asarray(action, dtype=np.float32), -1.0, 1.0)

            self.vel = 0.85 * self.vel + 0.15 * a
            new_pos = np.clip(self.pos + 0.06 * self.vel, -1.0, 1.0)

            if not self._blocked(new_pos):
                self.pos = new_pos
            else:
                self.vel *= 0.0

            dist = np.linalg.norm(self.goal - self.pos)
            reward = -float(dist)
            success = bool(dist < 0.10)
            terminated = success
            truncated = self.step_count >= self.max_steps
            return self._obs(), reward, terminated, truncated, {"success": success}

    env = PointMazeLiteEnv(max_steps=300)
    print("Using env: PointMazeLite-v0 (fallback)")
    return env, "PointMazeLite-v0"


def _extract_state(obs):
    arr = np.asarray(obs, dtype=np.float32)
    return arr if arr.ndim == 1 else arr.reshape(-1).astype(np.float32)


def _is_success(info, terminated):
    if isinstance(info, dict) and "success" in info:
        return bool(info["success"])
    return bool(terminated)


def _goal_distance_from_state(s):
    s = np.asarray(s, dtype=np.float32).reshape(-1)
    if s.shape[0] >= 6:
        # [x, y, vx, vy, goal_x, goal_y, ...]
        return float(np.linalg.norm(s[:2] - s[4:6]))
    return float("nan")


def evaluate_policy(
    policy_fn,
    env,
    num_episodes=20,
    max_steps=300,
    obs_window=2,
    action_scale=1.0,
    label="Policy",
    distance_success_threshold=0.20,
):
    successes = 0
    distance_successes = 0
    returns = []
    final_goal_dists = []
    min_goal_dists = []
    total_infer_time = 0.0
    total_steps = 0

    for ep in tqdm(range(num_episodes), desc=f"{label} eval"):
        obs, info = env.reset(seed=1000 + ep)
        s = _extract_state(obs)
        obs_hist = collections.deque([s.copy() for _ in range(obs_window)], maxlen=obs_window)

        ep_return = 0.0
        ep_success = False
        ep_min_dist = _goal_distance_from_state(s)

        for _ in range(max_steps):
            hist_np = np.stack(obs_hist, axis=0)
            t0 = time.time()
            action = policy_fn(hist_np)
            total_infer_time += (time.time() - t0)

            action = np.clip(np.asarray(action, dtype=np.float32) * action_scale, -1.0, 1.0)
            obs, reward, terminated, truncated, info = env.step(action)
            s = _extract_state(obs)
            obs_hist.append(s.copy())

            d = _goal_distance_from_state(s)
            if np.isfinite(d):
                ep_min_dist = min(ep_min_dist, d)

            ep_return += float(reward)
            total_steps += 1
            ep_success = ep_success or _is_success(info, terminated)
            if terminated or truncated:
                break

        final_dist = _goal_distance_from_state(s)
        final_goal_dists.append(final_dist)
        min_goal_dists.append(ep_min_dist)

        is_distance_success = np.isfinite(ep_min_dist) and (ep_min_dist <= distance_success_threshold)
        distance_successes += int(is_distance_success)

        successes += int(ep_success)
        returns.append(ep_return)

    return {
        "success_rate": 100.0 * successes / max(1, num_episodes),
        "num_successes": successes,
        "distance_success_rate": 100.0 * distance_successes / max(1, num_episodes),
        "distance_success_threshold": float(distance_success_threshold),
        "num_distance_successes": distance_successes,
        "num_episodes": num_episodes,
        "avg_return": float(np.mean(returns)) if returns else 0.0,
        "avg_final_goal_dist": float(np.nanmean(final_goal_dists)) if final_goal_dists else float("nan"),
        "avg_min_goal_dist": float(np.nanmean(min_goal_dists)) if min_goal_dists else float("nan"),
        "avg_step_time_ms": (1000.0 * total_infer_time / max(1, total_steps)),
    }


env, pointmaze_env_id = make_pointmaze_env()
liquid_result = evaluate_policy(liquid_pointmaze_action, env, num_episodes=20, max_steps=300, obs_window=PM_OBS_WINDOW, label="Liquid")
diffusion_result = evaluate_policy(diffusion_pointmaze_action, env, num_episodes=20, max_steps=300, obs_window=PM_OBS_WINDOW, label="Diffusion")
env.close()

print("\n" + "=" * 60)
print(f"Env: {pointmaze_env_id}")
print("POINTMAZE RESULTS")
print("=" * 60)
print(f"Liquid    success: {liquid_result['success_rate']:.2f}% ({liquid_result['num_successes']}/{liquid_result['num_episodes']})")
print(f"Diffusion success: {diffusion_result['success_rate']:.2f}% ({diffusion_result['num_successes']}/{diffusion_result['num_episodes']})")
print(
    f"Liquid distance success (<= {liquid_result['distance_success_threshold']:.2f}): "
    f"{liquid_result['distance_success_rate']:.2f}% "
    f"({liquid_result['num_distance_successes']}/{liquid_result['num_episodes']})"
)
print(
    f"Diff distance success   (<= {diffusion_result['distance_success_threshold']:.2f}): "
    f"{diffusion_result['distance_success_rate']:.2f}% "
    f"({diffusion_result['num_distance_successes']}/{diffusion_result['num_episodes']})"
)
print(f"Liquid avg return      : {liquid_result['avg_return']:.3f}")
print(f"Diff avg return        : {diffusion_result['avg_return']:.3f}")
print(f"Liquid avg final dist  : {liquid_result['avg_final_goal_dist']:.3f}")
print(f"Diff avg final dist    : {diffusion_result['avg_final_goal_dist']:.3f}")
print(f"Liquid avg min dist    : {liquid_result['avg_min_goal_dist']:.3f}")
print(f"Diff avg min dist      : {diffusion_result['avg_min_goal_dist']:.3f}")
print(f"Liquid step time       : {liquid_result['avg_step_time_ms']:.3f} ms")
print(f"Diff step time         : {diffusion_result['avg_step_time_ms']:.3f} ms")

In [ ]:
# PointMaze episode GIF (Liquid vs Diffusion side-by-side)

import io
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import Image, display

GOAL_POS = np.array([0.75, 0.75], dtype=np.float32)


def _render_pm_frame(pos, goal, step, ep_return, is_success, title):
    fig, ax = plt.subplots(figsize=(3.2, 3.2), dpi=80)
    ax.set_facecolor('#1a1a2e')
    fig.patch.set_facecolor('#1a1a2e')

    for sp in ax.spines.values():
        sp.set_edgecolor('#4a5568')
        sp.set_linewidth(1.5)

    wall_color = '#4a5568'
    for xy, w, h in [((-0.2, -0.8), 0.1, 1.2), ((0.1, -0.8), 0.1, 1.2), ((-0.2, -0.8), 0.4, 0.1)]:
        ax.add_patch(mpatches.Rectangle(xy, w, h, linewidth=0, facecolor=wall_color))

    ax.add_patch(plt.Circle(goal, 0.12, color='#f6e05e', alpha=0.25, zorder=4))
    ax.add_patch(plt.Circle(goal, 0.06, color='#f6e05e', alpha=0.90, zorder=5))
    ax.plot(*goal, marker='*', color='#744210', markersize=7, zorder=6)

    agent_color = '#68d391' if not is_success else '#f6ad55'
    ax.add_patch(plt.Circle(pos, 0.07, color=agent_color, alpha=1.0, zorder=10))
    ax.add_patch(plt.Circle(pos, 0.07, fill=False, edgecolor='white', linewidth=0.8, zorder=11))

    ax.set_xlim(-1.05, 1.05)
    ax.set_ylim(-1.05, 1.05)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])

    status = "✓ REACHED GOAL" if is_success else f"step {step}"
    title_color = '#f6e05e' if is_success else 'white'
    ax.set_title(f"{title} | {status}", color=title_color, fontsize=8, pad=3)
    ax.text(0.02, 0.02, f"return={ep_return:.1f}", transform=ax.transAxes, color='#a0aec0', fontsize=7, va='bottom')

    plt.tight_layout(pad=0.15)
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=80, facecolor=fig.get_facecolor())
    plt.close(fig)
    buf.seek(0)
    img = imageio.imread(buf)
    return img[:, :, :3]


def _run_episode_traj(policy_fn, obs_window, max_steps=300, seed=42):
    env, _ = make_pointmaze_env()
    obs, info = env.reset(seed=seed)
    s = _extract_state(obs)
    obs_hist = collections.deque([s.copy() for _ in range(obs_window)], maxlen=obs_window)

    traj, ep_return, is_success = [], 0.0, False
    for step in range(max_steps):
        traj.append((s[:2].copy(), ep_return, is_success, step))
        action = np.clip(np.asarray(policy_fn(np.stack(obs_hist, axis=0)), dtype=np.float32), -1.0, 1.0)
        obs, reward, terminated, truncated, info = env.step(action)
        s = _extract_state(obs)
        obs_hist.append(s.copy())
        ep_return += float(reward)
        if _is_success(info, terminated):
            is_success = True
        if terminated or truncated:
            break

    traj.append((s[:2].copy(), ep_return, is_success, step + 1))
    env.close()
    return traj


print("Recording PointMaze episodes for GIF…")
liq_traj = _run_episode_traj(liquid_pointmaze_action, obs_window=PM_OBS_WINDOW, seed=7)
diff_traj = _run_episode_traj(diffusion_pointmaze_action, obs_window=PM_OBS_WINDOW, seed=7)

max_len = max(len(liq_traj), len(diff_traj))
liq_traj += [liq_traj[-1]] * (max_len - len(liq_traj))
diff_traj += [diff_traj[-1]] * (max_len - len(diff_traj))

frames = []
for i in range(0, max_len, 3):
    lpos, lret, lsuc, lstep = liq_traj[i]
    dpos, dret, dsuc, dstep = diff_traj[i]
    lf = _render_pm_frame(lpos, GOAL_POS, lstep, lret, lsuc, "Liquid Net")
    df = _render_pm_frame(dpos, GOAL_POS, dstep, dret, dsuc, "Diffusion Policy")
    frames.append(np.concatenate([lf, df], axis=1))

gif_path = Path("artifacts") / "pointmaze_comparison.gif"
gif_path.parent.mkdir(parents=True, exist_ok=True)
imageio.mimsave(gif_path, frames, fps=8, loop=0)
print(f"Saved {gif_path} ({len(frames)} frames @ 8 fps)")
display(Image(filename=str(gif_path)))

## Sample-efficiency benchmark runner (training notebook)

This section runs **data-fraction scaling** for Push-T and PointMaze training in this notebook's environment, and writes a manifest for downstream closed-loop PointMaze evaluation in `pointmaze_eval.ipynb` (Python 3.12).

Outputs written to `artifacts/`:
- `sample_efficiency_training_manifest.json` (checkpoint + offline metrics per fraction/task)
- `sample_efficiency_training_summary.csv` (flat table for quick analysis)

In [ ]:
from torch.utils.data import DataLoader, Subset
from pathlib import Path
import json
import math
import time
import copy
import numpy as np
import pandas as pd

ARTIFACT_DIR = globals().get("ARTIFACT_DIR", "artifacts")

if "save_json" not in globals():
    def save_json(path, payload):
        p = Path(path)
        p.parent.mkdir(parents=True, exist_ok=True)
        with open(p, "w") as f:
            json.dump(payload, f, indent=2)

SAMPLE_EFF_LOG_MIN = 0.01
SAMPLE_EFF_LOG_MAX = 1.00
SAMPLE_EFF_LOG_POINTS = 7

def _make_log_fractions(min_frac=SAMPLE_EFF_LOG_MIN, max_frac=SAMPLE_EFF_LOG_MAX, n_points=SAMPLE_EFF_LOG_POINTS):
    vals = np.logspace(np.log10(float(min_frac)), np.log10(float(max_frac)), int(n_points))
    vals = [float(v) for v in vals]
    vals.append(1.0)  # ensure full-data endpoint
    vals = sorted(set(round(v, 6) for v in vals))
    return vals

SAMPLE_EFF_FRACTIONS = _make_log_fractions()
SAMPLE_EFF_SEED = 7
SAMPLE_EFF_EPOCHS = 120  # reduce if you want a faster smoke test
SAMPLE_EFF_TAG_PREFIX = "sampleeff"

SAMPLE_EFF_MANIFEST_PATH = Path(ARTIFACT_DIR) / "sample_efficiency_training_manifest.json"
SAMPLE_EFF_SUMMARY_CSV = Path(ARTIFACT_DIR) / "sample_efficiency_training_summary.csv"


def _pusht_fraction_loaders(frac: float):
    rng = np.random.default_rng(SAMPLE_EFF_SEED)
    train_indices = np.arange(len(train_dataset))
    rng.shuffle(train_indices)

    n_train = max(1, int(round(float(frac) * len(train_indices))))
    sub_train = Subset(train_dataset, train_indices[:n_train].tolist())

    tr = DataLoader(sub_train, batch_size=batch_size, shuffle=True, pin_memory=True)
    va = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)
    te = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

    state_dim = int(train_dataset[0]["state"].shape[-1])
    action_dim = int(train_dataset[0]["action"].shape[-1])
    return tr, va, te, state_dim, action_dim, n_train


def _pointmaze_fraction_loaders(frac: float):
    # Train subset by limiting max_train_episodes while keeping deterministic split logic in build_pointmaze_loaders().
    dataset = minari.load_dataset(POINTMAZE_DATASET_ID, download=True)
    total_eps = int(dataset.total_episodes)
    base_train_eps = max(1, int(round(0.80 * total_eps)))
    frac_train_eps = max(1, int(round(float(frac) * base_train_eps)))

    tr, va, te, meta = build_pointmaze_loaders(
        dataset_id=POINTMAZE_DATASET_ID,
        pred_horizon=16,
        obs_horizon=2,
        action_horizon=8,
        batch_size=POINTMAZE_BATCH_SIZE,
        seed=SAMPLE_EFF_SEED,
        max_train_episodes=frac_train_eps,
    )
    return tr, va, te, int(meta["state_dim"]), int(meta["action_dim"]), frac_train_eps


def run_sample_efficiency_training_benchmark(
    fractions=SAMPLE_EFF_FRACTIONS,
    epochs=SAMPLE_EFF_EPOCHS,
    patience=999,
    run_pusht=True,
    run_pointmaze=True,
):
    global MULTI_DATASET_TAG

    required_globals = ["MULTI_DATASET_TAG", "run_dataset_benchmark"]
    missing = [k for k in required_globals if k not in globals()]
    if missing:
        raise RuntimeError(
            "Missing setup symbols: " + ", ".join(missing) +
            ". Run the main setup/training-prep cells in liquid_jepa.ipynb first."
        )

    original_tag = MULTI_DATASET_TAG
    rows = []

    pointmaze_dataset_name = globals().get("POINTMAZE_NAME", "pointmaze_umaze")
    pointmaze_display_name = globals().get("POINTMAZE_LABEL", "pointmaze_umaze")

    try:
        for frac in fractions:
            frac_pct = int(round(100 * float(frac)))
            frac_bp = int(round(10000 * float(frac)))
            frac_token = f"f{frac_bp:05d}bp"
            frac_tag = f"{SAMPLE_EFF_TAG_PREFIX}_{frac_token}_seed{SAMPLE_EFF_SEED}"
            MULTI_DATASET_TAG = frac_tag

            print("\n" + "=" * 84)
            print(f"Sample-eff fraction={frac:.2%} | tag={frac_tag}")
            print("=" * 84)

            if run_pusht:
                print("\n[Push-T] Building fractional loaders...")
                tr, va, te, sdim, adim, n_train = _pusht_fraction_loaders(frac)
                print(f"[Push-T] Train samples used: {n_train}/{len(train_dataset)}")
                payload, _hist = run_dataset_benchmark(
                    dataset_name="pusht",
                    display_name="push_t",
                    train_loader=tr,
                    val_loader=va,
                    test_loader=te,
                    state_dim=sdim,
                    action_dim=adim,
                    num_epochs=epochs,
                    patience=patience,
                    batch_size=batch_size,
                )
                rows.append({
                    "task": "push_t",
                    "fraction": float(frac),
                    "fraction_pct": frac_pct,
                    "fraction_bp": frac_bp,
                    "n_train_units": int(n_train),
                    "tag": frac_tag,
                    "liquid_checkpoint": payload["liquid_checkpoint"],
                    "diffusion_checkpoint": payload["diffusion_checkpoint"],
                    "liquid_bestof10_mse": float(payload["liquid_results"]["k_metrics"]["10"]["best_of_k_mse"]),
                    "diffusion_bestof10_mse": float(payload["diffusion_results"]["k_metrics"]["10"]["best_of_k_mse"]),
                    "liquid_latency_ms": float(payload["liquid_results"]["single_pass_ms"]),
                    "diffusion_latency_ms": float(payload["diffusion_results"]["proxy_pass_ms"]),
                })

            if run_pointmaze:
                print("\n[PointMaze] Building fractional loaders...")
                tr, va, te, sdim, adim, n_train_eps = _pointmaze_fraction_loaders(frac)
                print(f"[PointMaze] Train episodes used: {n_train_eps}")
                payload, _hist = run_dataset_benchmark(
                    dataset_name=pointmaze_dataset_name,
                    display_name=pointmaze_display_name,
                    train_loader=tr,
                    val_loader=va,
                    test_loader=te,
                    state_dim=sdim,
                    action_dim=adim,
                    num_epochs=epochs,
                    patience=patience,
                    batch_size=POINTMAZE_BATCH_SIZE,
                )
                rows.append({
                    "task": "pointmaze",
                    "fraction": float(frac),
                    "fraction_pct": frac_pct,
                    "fraction_bp": frac_bp,
                    "n_train_units": int(n_train_eps),
                    "tag": frac_tag,
                    "liquid_checkpoint": payload["liquid_checkpoint"],
                    "diffusion_checkpoint": payload["diffusion_checkpoint"],
                    "liquid_bestof10_mse": float(payload["liquid_results"]["k_metrics"]["10"]["best_of_k_mse"]),
                    "diffusion_bestof10_mse": float(payload["diffusion_results"]["k_metrics"]["10"]["best_of_k_mse"]),
                    "liquid_latency_ms": float(payload["liquid_results"]["single_pass_ms"]),
                    "diffusion_latency_ms": float(payload["diffusion_results"]["proxy_pass_ms"]),
                })

    finally:
        MULTI_DATASET_TAG = original_tag

    rows_sorted = sorted(rows, key=lambda r: (r["task"], r["fraction"]))
    manifest = {
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "seed": int(SAMPLE_EFF_SEED),
        "fractions": [float(f) for f in fractions],
        "epochs": int(epochs),
        "rows": rows_sorted,
    }
    save_json(str(SAMPLE_EFF_MANIFEST_PATH), manifest)

    df = pd.DataFrame(rows_sorted)
    if len(df) > 0:
        df.to_csv(SAMPLE_EFF_SUMMARY_CSV, index=False)

    print("\n✓ Sample-efficiency training benchmark complete")
    print(f"  Manifest: {SAMPLE_EFF_MANIFEST_PATH}")
    print(f"  Summary : {SAMPLE_EFF_SUMMARY_CSV}")
    return manifest


print("Sample-efficiency training runner ready.")
print(f"Log fractions: {SAMPLE_EFF_FRACTIONS}")
print("Run:")
print("  run_sample_efficiency_training_benchmark()")
print("or quick smoke test:")
print("  run_sample_efficiency_training_benchmark(fractions=[0.1, 1.0], epochs=10)")


In [ ]:
# Full logarithmic-fraction training run (writes per-fraction checkpoints + manifest)
sampleeff_manifest = run_sample_efficiency_training_benchmark(
    fractions=SAMPLE_EFF_FRACTIONS,
    epochs=SAMPLE_EFF_EPOCHS,
    run_pusht=True,
    run_pointmaze=True,
 )
print(f"Rows written: {len(sampleeff_manifest['rows'])}")
print(f"Manifest: {SAMPLE_EFF_MANIFEST_PATH}")